# OP'26 Analytics — Agentic AI-Based Dynamic Tariff Optimization for EV Charging Networks

**Open Project 2026 — Society of Business**

This is a **self-contained, pre-executed notebook** that houses the entire codebase of the Agentic AI pricing engine. it can be run locally or directly in Google Colab.

### Pipeline Components:
1. **Shared Utilities & Styling** (Matplotlib dark-mode configuration)
2. **Data Preprocessing** (ACN-Data and UrbanEV telemetry parsing)
3. **Exploratory Data Analysis (EDA)** (14 dynamic visualizations)
4. **Demand Prediction Agent** (Random Forest and XGBoost forecasting models)
5. **Tariff Pricing Agent** (Dynamic Pricing simulation with demand elasticity)
6. **Monitoring & Learning Agent** (Closed-loop feedback parameter tuning)

In [8]:
# ==================== 1. SHARED UTILITIES & PATHS ====================

"""
Shared utilities, constants, and path configuration for OP'26 Analytics.
Agentic AI-Based Dynamic Tariff Optimization for EV Charging Networks.
"""

import os
import sys
import warnings

# Fix Windows console encoding
if sys.stdout and sys.stdout.encoding and sys.stdout.encoding.lower() != 'utf-8':
    try:
        sys.stdout.reconfigure(encoding='utf-8', errors='replace')
        sys.stderr.reconfigure(encoding='utf-8', errors='replace')
    except Exception:
        pass

import matplotlib
matplotlib.use('Agg')  # Non-interactive backend for plot generation
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

# ──────────────────────────────────────────────────────────────
# PATH CONFIGURATION
# ──────────────────────────────────────────────────────────────
# PROJECT_ROOT configuration (handles running inside Jupyter notebooks/exec where __file__ is undefined)
try:
    PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
except NameError:
    PROJECT_ROOT = os.getcwd()


# Find actual dataset paths dynamically (handles special characters)
_datasets_parent = None
for item in os.listdir(PROJECT_ROOT):
    if item.startswith("Datasets") and os.path.isdir(os.path.join(PROJECT_ROOT, item)):
        _inner = os.path.join(PROJECT_ROOT, item)
        for sub in os.listdir(_inner):
            if os.path.isdir(os.path.join(_inner, sub)):
                _datasets_parent = os.path.join(_inner, sub)
                break
        break

# ACN Data paths
ACN_DIR = None
URBANEV_DIR = None
if _datasets_parent:
    for item in os.listdir(_datasets_parent):
        full_path = os.path.join(_datasets_parent, item)
        if "ACN" in item and os.path.isdir(full_path):
            ACN_DIR = full_path
        elif "UrbanEV" in item and os.path.isdir(full_path):
            URBANEV_DIR = full_path

# Output paths
FIGURES_DIR = os.path.join(PROJECT_ROOT, "outputs", "figures")
METRICS_DIR = os.path.join(PROJECT_ROOT, "outputs", "metrics")

# Ensure output directories exist
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(METRICS_DIR, exist_ok=True)

# ──────────────────────────────────────────────────────────────
# ECONOMIC CONSTANTS
# ──────────────────────────────────────────────────────────────
BASELINE_TARIFF_PER_KWH = 15.0   # ₹15/kWh fixed baseline
SURGE_THRESHOLD = 0.80            # 80% utilization → surge pricing
DISCOUNT_THRESHOLD = 0.30         # 30% utilization → discount pricing
SURGE_MULTIPLIER_RANGE = (1.30, 1.80)
DISCOUNT_MULTIPLIER_RANGE = (0.60, 0.85)

# ──────────────────────────────────────────────────────────────
# PLOT STYLING — Professional, code-generated look
# ──────────────────────────────────────────────────────────────
PLOT_STYLE = {
    'figure.facecolor': '#0f1117',
    'axes.facecolor': '#1a1d29',
    'axes.edgecolor': '#2d3148',
    'axes.labelcolor': '#c8cdd5',
    'axes.grid': True,
    'grid.color': '#2d3148',
    'grid.alpha': 0.5,
    'grid.linestyle': '--',
    'text.color': '#c8cdd5',
    'xtick.color': '#8b92a5',
    'ytick.color': '#8b92a5',
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'legend.facecolor': '#1a1d29',
    'legend.edgecolor': '#2d3148',
    'legend.fontsize': 10,
    'figure.dpi': 150,
    'savefig.dpi': 150,
    'savefig.bbox': 'tight',
    'savefig.facecolor': '#0f1117',
    'savefig.pad_inches': 0.3,
}

# Color palettes
COLORS = {
    'primary': '#6366f1',      # Indigo
    'secondary': '#8b5cf6',    # Violet
    'accent': '#06b6d4',       # Cyan
    'success': '#10b981',      # Emerald
    'warning': '#f59e0b',      # Amber
    'danger': '#ef4444',       # Red
    'info': '#3b82f6',         # Blue
    'highlight': '#ec4899',    # Pink
    'muted': '#6b7280',        # Gray
}

PALETTE_SEQUENTIAL = ['#312e81', '#4338ca', '#6366f1', '#818cf8', '#a5b4fc', '#c7d2fe']
PALETTE_DIVERGING = ['#ef4444', '#f97316', '#facc15', '#4ade80', '#22d3ee', '#6366f1']
PALETTE_CATEGORICAL = ['#6366f1', '#06b6d4', '#10b981', '#f59e0b', '#ef4444', '#8b5cf6',
                        '#ec4899', '#14b8a6', '#f97316', '#3b82f6']


def apply_plot_style():
    """Apply the project's consistent dark-mode plot style."""
    plt.rcParams.update(PLOT_STYLE)


def save_figure(fig, filename, tight=True):
    """Save a matplotlib figure to the figures output directory."""
    filepath = os.path.join(FIGURES_DIR, filename)
    fig.savefig(filepath, bbox_inches='tight' if tight else None,
                facecolor=fig.get_facecolor(), edgecolor='none')
    plt.close(fig)
    print(f"  [SAVED] {filename}")
    return filepath


def save_metrics(df, filename):
    """Save a DataFrame to the metrics output directory as CSV."""
    filepath = os.path.join(METRICS_DIR, filename)
    df.to_csv(filepath, index=True)
    print(f"  [SAVED] {filename}")
    return filepath


def print_header(title):
    """Print a formatted section header."""
    width = 60
    print("\n" + "=" * width)
    print(f"  {title}")
    print("=" * width)


def print_subheader(title):
    """Print a formatted subsection header."""
    print(f"\n  -- {title} --")


In [9]:
# ==================== 2. DATA PREPROCESSING MODULE ====================

"""
Data Preprocessing Module
=========================
Loads, cleans, and engineers features from ACN-Data and UrbanEV datasets.
Creates a unified analytical base for downstream agents.
"""

import os
import pandas as pd
import numpy as np
from datetime import datetime
# from src.utils import (ACN_DIR, URBANEV_DIR, BASELINE_TARIFF_PER_KWH,
#                        print_header, print_subheader)


# ──────────────────────────────────────────────────────────────
# ACN DATA PREPROCESSING
# ──────────────────────────────────────────────────────────────

def load_acn_data():
    """Load the ACN charging sessions dataset."""
    print_subheader("Loading ACN-Data (Caltech/JPL sessions)")
    csv_path = os.path.join(ACN_DIR, "acndata_sessions.csv")
    df = pd.read_csv(csv_path, low_memory=False)
    print(f"  Raw records: {len(df):,}")
    print(f"  Columns: {list(df.columns)}")
    return df


def preprocess_acn(df):
    """
    Clean and feature-engineer the ACN dataset.
    
    Assumptions documented:
    - Rows with missing connectionTime or disconnectTime are dropped (essential timestamps)
    - kWhDelivered=0 sessions are retained (connection without charge is valid behavior)
    - Timezone is normalized to UTC for consistency
    """
    print_subheader("Preprocessing ACN-Data")
    
    # ── 1. Parse timestamps ──
    time_cols = ['connectionTime', 'disconnectTime', 'doneChargingTime']
    for col in time_cols:
        df[col] = pd.to_datetime(df[col], format='mixed', utc=True, errors='coerce')
    
    # ── 2. Drop rows missing critical timestamps ──
    n_before = len(df)
    df = df.dropna(subset=['connectionTime', 'disconnectTime'])
    n_dropped = n_before - len(df)
    print(f"  Dropped {n_dropped} rows with missing connection/disconnect times")
    
    # ── 3. Numeric conversion ──
    df['kWhDelivered'] = pd.to_numeric(df['kWhDelivered'], errors='coerce').fillna(0)
    df['clusterID'] = pd.to_numeric(df['clusterID'], errors='coerce')
    df['siteID'] = pd.to_numeric(df['siteID'], errors='coerce')
    
    # ── 4. Time-based feature engineering ──
    df['session_duration_hrs'] = (
        (df['disconnectTime'] - df['connectionTime']).dt.total_seconds() / 3600
    )
    
    # Charging duration (use doneChargingTime if available, else disconnectTime)
    charge_end = df['doneChargingTime'].fillna(df['disconnectTime'])
    df['charging_duration_hrs'] = (
        (charge_end - df['connectionTime']).dt.total_seconds() / 3600
    )
    
    # Idle time (plugged in but not charging)
    df['idle_time_hrs'] = df['session_duration_hrs'] - df['charging_duration_hrs']
    df['idle_time_hrs'] = df['idle_time_hrs'].clip(lower=0)
    
    # ── 5. Utilization & Revenue features ──
    df['charger_utilization_rate'] = np.where(
        df['session_duration_hrs'] > 0,
        df['charging_duration_hrs'] / df['session_duration_hrs'],
        0
    )
    df['charger_utilization_rate'] = df['charger_utilization_rate'].clip(0, 1)
    
    df['revenue_baseline'] = df['kWhDelivered'] * BASELINE_TARIFF_PER_KWH
    df['energy_cost_per_kwh'] = BASELINE_TARIFF_PER_KWH  # Fixed baseline
    
    # ── 6. Temporal features ──
    df['hour'] = df['connectionTime'].dt.hour
    df['day_of_week'] = df['connectionTime'].dt.dayofweek  # 0=Mon, 6=Sun
    df['day_name'] = df['connectionTime'].dt.day_name()
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    df['month'] = df['connectionTime'].dt.month
    df['date'] = df['connectionTime'].dt.date
    df['week'] = df['connectionTime'].dt.isocalendar().week.astype(int)
    
    # ── 7. Time-of-day period classification ──
    def classify_period(hour):
        if 7 <= hour <= 10 or 17 <= hour <= 20:
            return 'peak'
        elif 10 < hour < 17:
            return 'shoulder'
        else:
            return 'off-peak'
    
    df['period'] = df['hour'].apply(classify_period)
    
    # ── 8. Site labeling ──
    df['site_label'] = df['site'].fillna('unknown').str.strip().str.lower()
    
    # ── 9. Filter out physically impossible sessions ──
    n_before = len(df)
    df = df[df['session_duration_hrs'] > 0]
    df = df[df['session_duration_hrs'] < 48]  # Sessions > 48h are likely errors
    print(f"  Filtered {n_before - len(df)} impossible-duration sessions")
    
    # ── 10. Queue length proxy (sessions overlapping at same station) ──
    # Approximate: count sessions per station per hour
    df['station_hour_key'] = (
        df['stationID'].astype(str) + '_' + 
        df['connectionTime'].dt.strftime('%Y-%m-%d_%H')
    )
    queue_proxy = df.groupby('station_hour_key').size().rename('queue_length_proxy')
    df = df.merge(queue_proxy, on='station_hour_key', how='left')
    
    print(f"  Final ACN records: {len(df):,}")
    print(f"  Date range: {df['connectionTime'].min().date()} to {df['connectionTime'].max().date()}")
    print(f"  Sites: {df['site_label'].unique()}")
    print(f"  Unique stations: {df['stationID'].nunique()}")
    print(f"  Avg kWh/session: {df['kWhDelivered'].mean():.2f}")
    print(f"  Avg session duration: {df['session_duration_hrs'].mean():.2f} hrs")
    print(f"  Avg utilization rate: {df['charger_utilization_rate'].mean():.2%}")
    
    return df


# ──────────────────────────────────────────────────────────────
# URBANEV DATA PREPROCESSING
# ──────────────────────────────────────────────────────────────

def load_urbanev_data():
    """Load all UrbanEV (Shenzhen) dataset files."""
    print_subheader("Loading UrbanEV Dataset (Shenzhen)")
    
    data = {}
    
    # Time-series data (timestamp × grid matrices)
    ts_files = ['occupancy', 'price', 'duration', 'volume']
    for name in ts_files:
        filepath = os.path.join(URBANEV_DIR, f"{name}.csv")
        df = pd.read_csv(filepath)
        data[name] = df
        print(f"  {name}.csv: {df.shape[0]:,} timestamps × {df.shape[1]-1} grids")
    
    # Time index
    time_df = pd.read_csv(os.path.join(URBANEV_DIR, "time.csv"), encoding='utf-8-sig')
    data['time'] = time_df
    print(f"  time.csv: {len(time_df):,} entries")
    
    # Station metadata
    info_df = pd.read_csv(os.path.join(URBANEV_DIR, "information.csv"))
    data['information'] = info_df
    print(f"  information.csv: {len(info_df)} grids")
    
    stations_df = pd.read_csv(os.path.join(URBANEV_DIR, "stations.csv"))
    data['stations'] = stations_df
    print(f"  stations.csv: {len(stations_df):,} stations")
    
    # Spatial data
    adj_df = pd.read_csv(os.path.join(URBANEV_DIR, "adj.csv"))
    data['adj'] = adj_df
    print(f"  adj.csv: {adj_df.shape} adjacency matrix")
    
    dist_df = pd.read_csv(os.path.join(URBANEV_DIR, "distance.csv"))
    data['distance'] = dist_df
    print(f"  distance.csv: {dist_df.shape} distance matrix")
    
    return data


def preprocess_urbanev(data):
    """
    Clean and feature-engineer the UrbanEV dataset.
    
    Assumptions documented:
    - 5-minute interval timestamps from June 19, 2022 for 30 days (8,640 intervals)
    - Grid IDs correspond to Shenzhen district zones
    - Occupancy represents number of EVs actively charging
    - Missing values in time-series are forward-filled then backward-filled
    """
    print_subheader("Preprocessing UrbanEV Data")
    
    # ── 1. Build datetime index ──
    time_df = data['time']
    timestamps = pd.to_datetime(
        time_df.apply(
            lambda r: f"{int(r['year'])}-{int(r['month']):02d}-{int(r['day']):02d} "
                      f"{int(r['hour']):02d}:{int(r['minute']):02d}:{int(r['second']):02d}",
            axis=1
        )
    )
    print(f"  Time range: {timestamps.iloc[0]} to {timestamps.iloc[-1]}")
    print(f"  Interval: 5 minutes, {len(timestamps):,} steps")
    
    # ── 2. Process time-series with datetime index ──
    ts_names = ['occupancy', 'price', 'duration', 'volume']
    for name in ts_names:
        df = data[name].copy()
        # First column is timestamp index (numeric), replace with actual datetime
        idx_col = df.columns[0]
        df = df.drop(columns=[idx_col])
        df.index = timestamps
        df.index.name = 'timestamp'
        
        # Handle missing values: forward-fill then backward-fill
        n_missing = df.isna().sum().sum()
        if n_missing > 0:
            print(f"  {name}: {n_missing} missing values → ffill+bfill")
            df = df.ffill().bfill()
        
        data[name] = df
    
    # ── 3. Enrich grid information ──
    info = data['information'].copy()
    info['grid'] = info['grid'].astype(str)
    info['fast_ratio'] = info['fast_count'] / info['count'].replace(0, np.nan)
    info['fast_ratio'] = info['fast_ratio'].fillna(0)
    info['charger_density'] = info['count'] / info['area'].replace(0, np.nan)
    info['charger_density'] = info['charger_density'].fillna(0)
    data['information'] = info
    
    # ── 4. Compute derived grid-level time-series features ──
    grid_ids = [str(c) for c in data['occupancy'].columns]
    
    # Occupancy rate = occupancy / total chargers in grid
    occ = data['occupancy']
    grid_charger_count = info.set_index('grid')['count']
    
    occ_rate = occ.copy()
    for col in occ.columns:
        col_str = str(col)
        total = grid_charger_count.get(col_str, np.nan)
        if pd.notna(total) and total > 0:
            occ_rate[col] = occ[col] / total
        else:
            occ_rate[col] = 0
    occ_rate = occ_rate.clip(0, 1)
    data['occupancy_rate'] = occ_rate
    
    # ── 5. Temporal features for UrbanEV ──
    data['timestamps'] = timestamps
    
    # ── 6. Aggregate statistics ──
    avg_occ = occ.mean().mean()
    avg_price = data['price'].mean().mean()
    avg_vol = data['volume'].mean().mean()
    print(f"  Grid count: {len(grid_ids)}")
    print(f"  Avg occupancy across all grids: {avg_occ:.2f}")
    print(f"  Avg price across all grids: {avg_price:.4f}")
    print(f"  Avg volume across all grids: {avg_vol:.2f}")
    print(f"  CBD grids: {info['CBD'].sum()}, Non-CBD: {(~info['CBD'].astype(bool)).sum()}")
    print(f"  Dynamic pricing grids: {info['dynamic_pricing'].sum()}")
    
    return data


# ──────────────────────────────────────────────────────────────
# UNIFIED PREPROCESSING PIPELINE
# ──────────────────────────────────────────────────────────────

def run_preprocessing():
    """Execute the complete data preprocessing pipeline."""
    print_header("DATA PREPROCESSING")
    
    # ACN Data
    acn_raw = load_acn_data()
    acn_df = preprocess_acn(acn_raw)
    
    # UrbanEV Data
    urbanev_raw = load_urbanev_data()
    urbanev_data = preprocess_urbanev(urbanev_raw)
    
    print_subheader("Preprocessing Complete")
    print(f"  ACN: {len(acn_df):,} sessions ready")
    print(f"  UrbanEV: {len(urbanev_data['occupancy'])} timestamps × "
          f"{len(urbanev_data['occupancy'].columns)} grids ready")
    
    return acn_df, urbanev_data


if __name__ == "__main__":
    acn_df, urbanev_data = run_preprocessing()



  DATA PREPROCESSING

  -- Loading ACN-Data (Caltech/JPL sessions) --
  Raw records: 16,304
  Columns: ['_meta', 'end', 'min_kWh', 'site', 'start', '_items', '_id', 'clusterID', 'connectionTime', 'disconnectTime', 'doneChargingTime', 'kWhDelivered', 'sessionID', 'siteID', 'spaceID', 'stationID', 'timezone', 'userID', 'userInputs', 'WhPerMile', 'kWhRequested', 'milesRequested', 'minutesAvailable', 'modifiedAt', 'paymentRequired', 'requestedDeparture', 'userID.1']

  -- Preprocessing ACN-Data --
  Dropped 1305 rows with missing connection/disconnect times
  Filtered 52 impossible-duration sessions
  Final ACN records: 14,947
  Date range: 2018-04-25 to 2018-12-16
  Sites: ['caltech' 'unknown']
  Unique stations: 54
  Avg kWh/session: 9.00
  Avg session duration: 5.68 hrs
  Avg utilization rate: 69.34%

  -- Loading UrbanEV Dataset (Shenzhen) --
  occupancy.csv: 8,640 timestamps × 247 grids
  price.csv: 8,640 timestamps × 247 grids
  duration.csv: 8,640 timestamps × 247 grids
  volume.cs

In [10]:
# ==================== 3. EXPLORATORY DATA ANALYSIS ====================

"""
Exploratory Data Analysis — OP'26 Analytics
Agentic AI-Based Dynamic Tariff Optimization for EV Charging Networks.

Generates 14 publication-quality dark-mode plots covering:
  • ACN session demand, utilization, revenue, and idle-time patterns
  • UrbanEV occupancy heatmaps, price-demand relationships, and spatial analysis
  • Combined comparative views of both datasets
"""

# from src.utils import (
#     apply_plot_style, save_figure, FIGURES_DIR, COLORS,
#     PALETTE_SEQUENTIAL, PALETTE_DIVERGING, PALETTE_CATEGORICAL,
#     print_header, print_subheader,
# )
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
import seaborn as sns
import pandas as pd
import numpy as np
from matplotlib.lines import Line2D
from matplotlib.patches import Patch


# ── colour maps keyed to period labels ──────────────────────────
PERIOD_COLORS = {
    'peak':      COLORS['danger'],      # red
    'shoulder':  COLORS['warning'],     # amber
    'off-peak':  COLORS['success'],     # emerald
}

SITE_COLORS = {
    'caltech': PALETTE_CATEGORICAL[0],  # indigo
    'jpl':     PALETTE_CATEGORICAL[1],  # cyan
}

DAY_ORDER = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday',
             'Saturday', 'Sunday']


# ═════════════════════════════════════════════════════════════════
#  HELPER UTILITIES
# ═════════════════════════════════════════════════════════════════

def _annotate_bar_values(ax, fmt='{:.1f}', fontsize=8, color='#c8cdd5',
                         offset=0.02):
    """Add value labels above every bar in *ax*."""
    ymax = ax.get_ylim()[1]
    for p in ax.patches:
        h = p.get_height()
        if np.isnan(h) or h == 0:
            continue
        ax.annotate(fmt.format(h),
                    (p.get_x() + p.get_width() / 2., h),
                    ha='center', va='bottom',
                    fontsize=fontsize, color=color,
                    xytext=(0, 3), textcoords='offset points')


def _period_for_hour(h):
    """Return 'peak', 'shoulder', or 'off-peak' for a given hour."""
    if h in range(7, 11) or h in range(17, 21):
        return 'peak'
    elif h in range(11, 17):
        return 'shoulder'
    else:
        return 'off-peak'


# ═════════════════════════════════════════════════════════════════
#  PLOT 01 — ACN Hourly Demand (bar by period)
# ═════════════════════════════════════════════════════════════════

def _plot_01_hourly_demand(acn_df):
    hourly = acn_df.groupby('hour').size().reindex(range(24), fill_value=0)
    # Normalise to mean sessions per day
    n_days = max(acn_df['date'].nunique(), 1)
    hourly_avg = hourly / n_days

    colors = [PERIOD_COLORS[_period_for_hour(h)] for h in range(24)]

    fig, ax = plt.subplots(figsize=(12, 5))
    bars = ax.bar(range(24), hourly_avg.values, color=colors, edgecolor='none',
                  width=0.75, zorder=3)
    ax.set_xlabel('Hour of Day')
    ax.set_ylabel('Avg Sessions per Day')
    ax.set_title('ACN — Hourly Charging Demand by Time-of-Use Period',
                 fontsize=15, fontweight='bold', pad=12)
    ax.set_xticks(range(24))
    ax.set_xticklabels([f'{h:02d}' for h in range(24)], fontsize=9)
    ax.set_xlim(-0.6, 23.6)

    # legend
    handles = [Patch(facecolor=PERIOD_COLORS[p], label=p.title())
               for p in ('peak', 'shoulder', 'off-peak')]
    ax.legend(handles=handles, loc='upper right', framealpha=0.85)

    # annotate peak hour
    peak_h = int(hourly_avg.idxmax())
    ax.annotate(f'Peak: {hourly_avg.max():.1f}',
                xy=(peak_h, hourly_avg.max()),
                xytext=(peak_h + 2, hourly_avg.max() * 1.12),
                arrowprops=dict(arrowstyle='->', color=COLORS['accent'],
                                lw=1.5),
                fontsize=10, color=COLORS['accent'], fontweight='bold')

    fig.tight_layout()
    save_figure(fig, '01_acn_hourly_demand.png')


# ═════════════════════════════════════════════════════════════════
#  PLOT 02 — ACN Weekday Pattern (grouped by site)
# ═════════════════════════════════════════════════════════════════

def _plot_02_weekday_pattern(acn_df):
    ct = (acn_df.groupby(['day_of_week', 'day_name', 'site_label'])
          .size().reset_index(name='sessions'))
    n_weeks = max(acn_df['week'].nunique(), 1)
    ct['avg_sessions'] = ct['sessions'] / n_weeks

    pivot = ct.pivot_table(index='day_of_week', columns='site_label',
                           values='avg_sessions', fill_value=0)
    pivot = pivot.reindex(range(7))

    fig, ax = plt.subplots(figsize=(10, 5))
    x = np.arange(7)
    w = 0.35
    for i, site in enumerate(('caltech', 'jpl')):
        if site in pivot.columns:
            vals = pivot[site].values
            ax.bar(x + i * w - w / 2, vals, w, label=site.upper(),
                   color=SITE_COLORS[site], edgecolor='none', zorder=3)

    ax.set_xticks(x)
    ax.set_xticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'],
                       fontsize=10)
    ax.set_xlabel('Day of Week')
    ax.set_ylabel('Avg Sessions per Week')
    ax.set_title('ACN — Weekly Demand Pattern by Site',
                 fontsize=15, fontweight='bold', pad=12)
    ax.legend(framealpha=0.85)

    # weekend shading
    ax.axvspan(4.5, 6.5, color='white', alpha=0.04, zorder=0)
    ax.text(5.5, ax.get_ylim()[1] * 0.92, 'Weekend', ha='center',
            fontsize=9, color=COLORS['muted'], style='italic')

    fig.tight_layout()
    save_figure(fig, '02_acn_weekday_pattern.png')


# ═════════════════════════════════════════════════════════════════
#  PLOT 03 — kWh Distribution (histogram + KDE)
# ═════════════════════════════════════════════════════════════════

def _plot_03_kwh_distribution(acn_df):
    data = acn_df['kWhDelivered'].dropna()
    mean_v, med_v = data.mean(), data.median()
    std_v = data.std()

    fig, ax = plt.subplots(figsize=(10, 5))
    sns.histplot(data, bins=60, kde=True, color=COLORS['primary'],
                 edgecolor='none', alpha=0.7, ax=ax, stat='density',
                 line_kws={'lw': 2})

    ax.axvline(mean_v, color=COLORS['warning'], ls='--', lw=1.8,
               label=f'Mean = {mean_v:.2f} kWh')
    ax.axvline(med_v, color=COLORS['accent'], ls='--', lw=1.8,
               label=f'Median = {med_v:.2f} kWh')

    # stats box
    stats_text = (f'μ = {mean_v:.2f} kWh\n'
                  f'σ = {std_v:.2f} kWh\n'
                  f'n = {len(data):,}')
    ax.text(0.97, 0.95, stats_text, transform=ax.transAxes,
            ha='right', va='top', fontsize=10, color='#c8cdd5',
            bbox=dict(boxstyle='round,pad=0.5', fc='#1a1d29',
                      ec='#2d3148', alpha=0.9))

    ax.set_xlabel('Energy Delivered (kWh)')
    ax.set_ylabel('Density')
    ax.set_title('ACN — Distribution of Energy Delivered per Session',
                 fontsize=15, fontweight='bold', pad=12)
    ax.legend(loc='upper left', framealpha=0.85)
    fig.tight_layout()
    save_figure(fig, '03_acn_kwh_distribution.png')


# ═════════════════════════════════════════════════════════════════
#  PLOT 04 — Session Duration by Period (violin + box)
# ═════════════════════════════════════════════════════════════════

def _plot_04_session_duration(acn_df):
    order = ['peak', 'shoulder', 'off-peak']
    pal = [PERIOD_COLORS[p] for p in order]

    fig, ax = plt.subplots(figsize=(9, 6))
    sns.violinplot(data=acn_df, x='period', y='session_duration_hrs',
                   order=order, palette=pal, inner=None, alpha=0.35,
                   cut=0, ax=ax, zorder=2)
    sns.boxplot(data=acn_df, x='period', y='session_duration_hrs',
                order=order, palette=pal, width=0.2, linewidth=1.2,
                fliersize=2, ax=ax, zorder=3,
                boxprops=dict(alpha=0.9),
                medianprops=dict(color='white', lw=1.5))

    ax.set_xlabel('Time-of-Use Period')
    ax.set_ylabel('Session Duration (hours)')
    ax.set_title('ACN — Session Duration by TOU Period',
                 fontsize=15, fontweight='bold', pad=12)
    ax.set_xticklabels([p.title() for p in order])

    # annotate medians
    for i, period in enumerate(order):
        subset = acn_df.loc[acn_df['period'] == period, 'session_duration_hrs']
        med = subset.median()
        ax.text(i, med + 0.3, f'{med:.1f}h', ha='center', fontsize=9,
                color='white', fontweight='bold')

    fig.tight_layout()
    save_figure(fig, '04_acn_session_duration.png')


# ═════════════════════════════════════════════════════════════════
#  PLOT 05 — Utilization Heatmap (hour × day_of_week)
# ═════════════════════════════════════════════════════════════════

def _plot_05_utilization_heatmap(acn_df):
    pivot = acn_df.pivot_table(index='hour', columns='day_of_week',
                               values='charger_utilization_rate',
                               aggfunc='mean')
    pivot = pivot.reindex(index=range(24), columns=range(7))

    fig, ax = plt.subplots(figsize=(9, 8))
    cmap = sns.color_palette(PALETTE_SEQUENTIAL, as_cmap=True)
    sns.heatmap(pivot, cmap='YlOrRd', ax=ax, linewidths=0.3,
                linecolor='#0f1117', cbar_kws={'label': 'Mean Utilization',
                                                'shrink': 0.8},
                annot=True, fmt='.2f', annot_kws={'fontsize': 7})

    ax.set_yticklabels([f'{h:02d}:00' for h in range(24)], rotation=0,
                       fontsize=8)
    ax.set_xticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'],
                       fontsize=9)
    ax.set_xlabel('Day of Week')
    ax.set_ylabel('Hour of Day')
    ax.set_title('ACN — Charger Utilization Heatmap',
                 fontsize=15, fontweight='bold', pad=12)
    fig.tight_layout()
    save_figure(fig, '05_acn_utilization_heatmap.png')


# ═════════════════════════════════════════════════════════════════
#  PLOT 06 — Revenue by Period (stacked bar by date)
# ═════════════════════════════════════════════════════════════════

def _plot_06_revenue_by_period(acn_df):
    rev = (acn_df.groupby(['date', 'period'])['revenue_baseline']
           .sum().reset_index())
    rev_pivot = rev.pivot_table(index='date', columns='period',
                                values='revenue_baseline', fill_value=0)
    order = ['off-peak', 'shoulder', 'peak']
    rev_pivot = rev_pivot.reindex(columns=order, fill_value=0)
    rev_pivot = rev_pivot.sort_index()

    fig, ax = plt.subplots(figsize=(13, 5))
    bottom = np.zeros(len(rev_pivot))
    for period in order:
        if period not in rev_pivot.columns:
            continue
        vals = rev_pivot[period].values
        ax.bar(range(len(rev_pivot)), vals, bottom=bottom,
               color=PERIOD_COLORS[period], label=period.title(),
               edgecolor='none', width=1.0, zorder=3)
        bottom += vals

    # x-axis: show every Nth date
    n = max(len(rev_pivot) // 15, 1)
    tick_pos = list(range(0, len(rev_pivot), n))
    tick_labels = [str(rev_pivot.index[i]) for i in tick_pos]
    ax.set_xticks(tick_pos)
    ax.set_xticklabels(tick_labels, rotation=45, ha='right', fontsize=8)

    ax.set_xlabel('Date')
    ax.set_ylabel('Revenue (₹)')
    ax.set_title('ACN — Daily Baseline Revenue by TOU Period',
                 fontsize=15, fontweight='bold', pad=12)
    ax.legend(loc='upper left', framealpha=0.85)

    # total annotation
    total_rev = rev_pivot.sum().sum()
    ax.text(0.98, 0.95, f'Total: ₹{total_rev:,.0f}',
            transform=ax.transAxes, ha='right', va='top', fontsize=11,
            color=COLORS['accent'], fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.4', fc='#1a1d29',
                      ec='#2d3148', alpha=0.9))

    fig.tight_layout()
    save_figure(fig, '06_acn_revenue_by_period.png')


# ═════════════════════════════════════════════════════════════════
#  PLOT 07 — Idle Time vs kWh (scatter by period)
# ═════════════════════════════════════════════════════════════════

def _plot_07_idle_time(acn_df):
    df = acn_df.dropna(subset=['idle_time_hrs', 'kWhDelivered']).copy()

    fig, ax = plt.subplots(figsize=(10, 6))
    for period in ('peak', 'shoulder', 'off-peak'):
        sub = df[df['period'] == period]
        ax.scatter(sub['kWhDelivered'], sub['idle_time_hrs'],
                   c=PERIOD_COLORS[period], label=period.title(),
                   alpha=0.35, s=12, edgecolors='none', zorder=3)

    ax.set_xlabel('Energy Delivered (kWh)')
    ax.set_ylabel('Idle Time (hours)')
    ax.set_title('ACN — Idle Time vs Energy Delivered',
                 fontsize=15, fontweight='bold', pad=12)
    ax.legend(framealpha=0.85, markerscale=3)

    # correlation annotation
    corr = df[['idle_time_hrs', 'kWhDelivered']].corr().iloc[0, 1]
    ax.text(0.97, 0.05, f'ρ = {corr:.3f}',
            transform=ax.transAxes, ha='right', va='bottom',
            fontsize=11, color=COLORS['accent'], fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.4', fc='#1a1d29',
                      ec='#2d3148', alpha=0.9))

    fig.tight_layout()
    save_figure(fig, '07_acn_idle_time_analysis.png')


# ═════════════════════════════════════════════════════════════════
#  PLOT 08 — UrbanEV Demand Heatmap (hour × top-20 grids)
# ═════════════════════════════════════════════════════════════════

def _plot_08_urbanev_heatmap(urbanev_data):
    occ = urbanev_data['occupancy']
    # Total occupancy per grid → top 20
    top_grids = occ.sum().nlargest(20).index.tolist()
    occ_top = occ[top_grids].copy()
    occ_top['hour'] = occ_top.index.hour
    hourly_avg = occ_top.groupby('hour').mean()

    fig, ax = plt.subplots(figsize=(14, 7))
    sns.heatmap(hourly_avg.T, cmap='magma', ax=ax, linewidths=0.3,
                linecolor='#0f1117',
                cbar_kws={'label': 'Avg Charging Pile Count', 'shrink': 0.8})

    ax.set_xlabel('Hour of Day')
    ax.set_ylabel('Grid ID')
    ax.set_title('UrbanEV — Hourly Demand Heatmap (Top 20 Grids)',
                 fontsize=15, fontweight='bold', pad=12)
    ax.set_xticklabels([f'{h:02d}' for h in range(24)], fontsize=8)
    ax.set_yticklabels(ax.get_yticklabels(), fontsize=7, rotation=0)
    fig.tight_layout()
    save_figure(fig, '08_urbanev_demand_heatmap.png')


# ═════════════════════════════════════════════════════════════════
#  PLOT 09 — Price vs Demand Scatter (by CBD flag)
# ═════════════════════════════════════════════════════════════════

def _plot_09_price_demand(urbanev_data):
    occ_rate = urbanev_data['occupancy_rate']
    price = urbanev_data['price']
    info = urbanev_data['information']

    common_grids = list(set(occ_rate.columns) & set(price.columns))
    avg_occ = occ_rate[common_grids].mean()
    avg_price = price[common_grids].mean()

    scatter_df = pd.DataFrame({'avg_occupancy_rate': avg_occ,
                                'avg_price': avg_price}).dropna()

    # Merge CBD flag
    if 'grid' in info.columns and 'CBD' in info.columns:
        cbd_map = info.set_index('grid')['CBD']
        # Convert grid column types to match
        scatter_df['CBD'] = scatter_df.index.map(
            lambda g: cbd_map.get(g, cbd_map.get(str(g), 0)))
    else:
        scatter_df['CBD'] = 0

    scatter_df['CBD'] = scatter_df['CBD'].fillna(0).astype(int)

    fig, ax = plt.subplots(figsize=(10, 6))
    for label, color, marker in [(1, COLORS['warning'], 'D'),
                                  (0, COLORS['primary'], 'o')]:
        sub = scatter_df[scatter_df['CBD'] == label]
        ax.scatter(sub['avg_price'], sub['avg_occupancy_rate'],
                   c=color, label=f'CBD = {label}', alpha=0.7,
                   s=50, edgecolors='white', linewidths=0.3,
                   marker=marker, zorder=3)

    # correlation line
    valid = scatter_df.dropna()
    if len(valid) > 2:
        corr = valid[['avg_price', 'avg_occupancy_rate']].corr().iloc[0, 1]
        z = np.polyfit(valid['avg_price'], valid['avg_occupancy_rate'], 1)
        p = np.poly1d(z)
        xr = np.linspace(valid['avg_price'].min(), valid['avg_price'].max(), 50)
        ax.plot(xr, p(xr), '--', color=COLORS['accent'], lw=1.5, alpha=0.7)
        ax.text(0.97, 0.05, f'ρ = {corr:.3f}',
                transform=ax.transAxes, ha='right', va='bottom',
                fontsize=11, color=COLORS['accent'], fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.4', fc='#1a1d29',
                          ec='#2d3148', alpha=0.9))

    ax.set_xlabel('Average Price (¥/kWh)')
    ax.set_ylabel('Average Occupancy Rate')
    ax.set_title('UrbanEV — Price vs Occupancy Rate by CBD Status',
                 fontsize=15, fontweight='bold', pad=12)
    ax.legend(framealpha=0.85)
    fig.tight_layout()
    save_figure(fig, '09_urbanev_price_demand_scatter.png')


# ═════════════════════════════════════════════════════════════════
#  PLOT 10 — Weekly Temporal Profile
# ═════════════════════════════════════════════════════════════════

def _plot_10_temporal_profile(urbanev_data):
    occ_rate = urbanev_data['occupancy_rate']
    mean_profile = occ_rate.mean(axis=1)
    profile_df = pd.DataFrame({'occupancy_rate': mean_profile})
    profile_df['hour'] = profile_df.index.hour
    profile_df['dow'] = profile_df.index.dayofweek  # 0=Mon
    profile_df['is_weekend'] = profile_df['dow'].isin([5, 6]).astype(int)

    # Build a typical-week profile: 7 days × 24 hours
    weekly = profile_df.groupby(['dow', 'hour'])['occupancy_rate'].mean()
    weekly = weekly.reindex(pd.MultiIndex.from_product(
        [range(7), range(24)], names=['dow', 'hour']), fill_value=np.nan)

    x = np.arange(7 * 24)
    y = weekly.values

    fig, ax = plt.subplots(figsize=(14, 5))

    # shade weekends
    for start in [5 * 24, 6 * 24]:
        ax.axvspan(start, start + 24, color='white', alpha=0.04, zorder=0)

    ax.fill_between(x, y, alpha=0.25, color=COLORS['primary'], zorder=2)
    ax.plot(x, y, color=COLORS['primary'], lw=1.5, zorder=3)

    # day boundaries
    for d in range(1, 7):
        ax.axvline(d * 24, color='#2d3148', ls=':', lw=0.8, zorder=1)

    ax.set_xticks([d * 24 + 12 for d in range(7)])
    ax.set_xticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'],
                       fontsize=10)
    ax.set_xlim(0, 7 * 24 - 1)
    ax.set_xlabel('Day of Week')
    ax.set_ylabel('Mean Occupancy Rate')
    ax.set_title('UrbanEV — Typical Weekly Occupancy Profile',
                 fontsize=15, fontweight='bold', pad=12)

    # annotate weekday vs weekend means
    wd_mean = profile_df.loc[profile_df['is_weekend'] == 0, 'occupancy_rate'].mean()
    we_mean = profile_df.loc[profile_df['is_weekend'] == 1, 'occupancy_rate'].mean()
    ax.text(0.02, 0.95,
            f'Weekday avg: {wd_mean:.3f}\nWeekend avg: {we_mean:.3f}',
            transform=ax.transAxes, ha='left', va='top', fontsize=10,
            color='#c8cdd5',
            bbox=dict(boxstyle='round,pad=0.5', fc='#1a1d29',
                      ec='#2d3148', alpha=0.9))

    fig.tight_layout()
    save_figure(fig, '10_urbanev_temporal_profile.png')


# ═════════════════════════════════════════════════════════════════
#  PLOT 11 — Station Map (scatter by lon/lat)
# ═════════════════════════════════════════════════════════════════

def _plot_11_station_map(urbanev_data):
    stations = urbanev_data['stations'].copy()
    # Ensure numeric types
    for col in ['longitude', 'latitude', 'fast', 'slow', 'count']:
        if col in stations.columns:
            stations[col] = pd.to_numeric(stations[col], errors='coerce')

    stations = stations.dropna(subset=['longitude', 'latitude', 'count'])

    fast_ratio = (stations['fast'] / stations['count'].replace(0, np.nan)).fillna(0)

    fig, ax = plt.subplots(figsize=(10, 8))
    sc = ax.scatter(stations['longitude'], stations['latitude'],
                    s=stations['count'] * 8, c=fast_ratio,
                    cmap='coolwarm', alpha=0.7, edgecolors='white',
                    linewidths=0.3, zorder=3, vmin=0, vmax=1)
    cbar = fig.colorbar(sc, ax=ax, shrink=0.7, pad=0.02)
    cbar.set_label('Fast Charger Ratio', fontsize=10)

    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title('UrbanEV — Charging Station Map (size ∝ charger count)',
                 fontsize=15, fontweight='bold', pad=12)

    # size legend
    for s_val in [10, 30, 60]:
        ax.scatter([], [], s=s_val * 8, c=COLORS['muted'], alpha=0.6,
                   edgecolors='white', linewidths=0.3,
                   label=f'{s_val} chargers')
    ax.legend(loc='lower left', framealpha=0.85, title='Count',
              title_fontsize=9)

    fig.tight_layout()
    save_figure(fig, '11_urbanev_station_map.png')


# ═════════════════════════════════════════════════════════════════
#  PLOT 12 — Peak vs Off-Peak Occupancy (top 15 grids)
# ═════════════════════════════════════════════════════════════════

def _plot_12_peak_offpeak(urbanev_data):
    occ_rate = urbanev_data['occupancy_rate'].copy()
    occ_rate_df = occ_rate.copy()
    occ_rate_df['hour'] = occ_rate_df.index.hour
    is_peak = occ_rate_df['hour'].isin(list(range(7, 11)) + list(range(17, 21)))

    # Top 15 grids by total occupancy
    grid_cols = [c for c in occ_rate.columns]
    top15 = occ_rate[grid_cols].sum().nlargest(15).index.tolist()

    peak_vals = occ_rate_df.loc[is_peak, top15].mean()
    offpeak_vals = occ_rate_df.loc[~is_peak, top15].mean()

    fig, ax = plt.subplots(figsize=(12, 6))
    x = np.arange(len(top15))
    w = 0.35

    ax.bar(x - w / 2, peak_vals.values, w, label='Peak (7-10, 17-20)',
           color=COLORS['danger'], edgecolor='none', zorder=3)
    ax.bar(x + w / 2, offpeak_vals.values, w, label='Off-Peak',
           color=COLORS['success'], edgecolor='none', zorder=3)

    ax.set_xticks(x)
    ax.set_xticklabels([str(g) for g in top15], rotation=45, ha='right',
                       fontsize=8)
    ax.set_xlabel('Grid ID')
    ax.set_ylabel('Mean Occupancy Rate')
    ax.set_title('UrbanEV — Peak vs Off-Peak Occupancy (Top 15 Grids)',
                 fontsize=15, fontweight='bold', pad=12)
    ax.legend(framealpha=0.85)

    fig.tight_layout()
    save_figure(fig, '12_urbanev_peak_offpeak.png')


# ═════════════════════════════════════════════════════════════════
#  PLOT 13 — Volatility Analysis (rolling std)
# ═════════════════════════════════════════════════════════════════

def _plot_13_volatility(urbanev_data):
    occ_rate = urbanev_data['occupancy_rate']
    overall_mean = occ_rate.mean(axis=1)

    top5 = occ_rate.sum().nlargest(5).index.tolist()

    window = min(24, len(occ_rate) // 4) or 24  # 24-hour rolling
    if window < 2:
        window = 2

    fig, ax = plt.subplots(figsize=(14, 5))

    # Overall mean volatility
    roll_overall = overall_mean.rolling(window, min_periods=1).std()
    ax.plot(roll_overall.index, roll_overall.values, color='white',
            lw=2.2, label='Overall Mean', zorder=4, alpha=0.9)

    # Top-5 grids
    for i, g in enumerate(top5):
        roll_g = occ_rate[g].rolling(window, min_periods=1).std()
        ax.plot(roll_g.index, roll_g.values,
                color=PALETTE_CATEGORICAL[i], lw=1.2, alpha=0.7,
                label=f'Grid {g}', zorder=3)

    ax.set_xlabel('Timestamp')
    ax.set_ylabel(f'Rolling Std (window={window}h)')
    ax.set_title('UrbanEV — Demand Volatility (Rolling Std of Occupancy Rate)',
                 fontsize=15, fontweight='bold', pad=12)
    ax.legend(loc='upper right', framealpha=0.85, fontsize=9)

    # Format x-axis dates
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d'))
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    fig.autofmt_xdate(rotation=30)

    fig.tight_layout()
    save_figure(fig, '13_volatility_analysis.png')


# ═════════════════════════════════════════════════════════════════
#  PLOT 14 — Combined Demand Patterns (2×2)
# ═════════════════════════════════════════════════════════════════

def _plot_14_combined(acn_df, urbanev_data):
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Combined — ACN vs UrbanEV Demand Patterns',
                 fontsize=16, fontweight='bold', y=0.98, color='#c8cdd5')

    occ_rate = urbanev_data['occupancy_rate']

    # ── (a) Hourly demand comparison ────────────────────────────
    ax = axes[0, 0]
    n_days_acn = max(acn_df['date'].nunique(), 1)
    acn_hourly = (acn_df.groupby('hour').size() / n_days_acn).reindex(
        range(24), fill_value=0)
    uev_hourly = occ_rate.groupby(occ_rate.index.hour).mean().mean(axis=1)
    uev_hourly = uev_hourly.reindex(range(24), fill_value=0)

    ax2 = ax.twinx()
    ax.bar(range(24), acn_hourly.values, color=COLORS['primary'],
           alpha=0.7, width=0.7, label='ACN (sessions)', zorder=3)
    ax2.plot(range(24), uev_hourly.values, color=COLORS['warning'],
             lw=2, marker='o', ms=4, label='UrbanEV (occ rate)', zorder=4)
    ax.set_xlabel('Hour', fontsize=9)
    ax.set_ylabel('ACN Sessions', fontsize=9, color=COLORS['primary'])
    ax2.set_ylabel('UrbanEV Occ Rate', fontsize=9, color=COLORS['warning'])
    ax.set_title('(a) Hourly Demand', fontsize=12, fontweight='bold')
    ax.set_xticks(range(0, 24, 3))

    # Combined legend
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, loc='upper left', fontsize=8,
              framealpha=0.85)

    # ── (b) Weekday pattern ─────────────────────────────────────
    ax = axes[0, 1]
    acn_dow = (acn_df.groupby('day_of_week').size() /
               max(acn_df['week'].nunique(), 1)).reindex(range(7), fill_value=0)

    uev_dow_df = occ_rate.copy()
    uev_dow_df['dow'] = uev_dow_df.index.dayofweek
    uev_dow = uev_dow_df.groupby('dow').mean().mean(axis=1).reindex(
        range(7), fill_value=0)

    x = np.arange(7)
    w = 0.35
    ax2 = ax.twinx()
    ax.bar(x - w / 2, acn_dow.values, w, color=COLORS['primary'],
           alpha=0.7, label='ACN', zorder=3)
    ax2.bar(x + w / 2, uev_dow.values, w, color=COLORS['warning'],
            alpha=0.7, label='UrbanEV', zorder=3)
    ax.set_xticks(x)
    ax.set_xticklabels(['M', 'T', 'W', 'T', 'F', 'S', 'S'], fontsize=9)
    ax.set_ylabel('ACN Avg Sessions', fontsize=9, color=COLORS['primary'])
    ax2.set_ylabel('UrbanEV Occ Rate', fontsize=9, color=COLORS['warning'])
    ax.set_title('(b) Weekday Pattern', fontsize=12, fontweight='bold')
    h1, l1 = ax.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, loc='upper right', fontsize=8,
              framealpha=0.85)

    # ── (c) Utilization / Occupancy distribution ────────────────
    ax = axes[1, 0]
    sns.kdeplot(acn_df['charger_utilization_rate'].dropna(), ax=ax,
                color=COLORS['primary'], fill=True, alpha=0.4, lw=2,
                label='ACN Utilization', clip=(0, 1))
    # Flatten UrbanEV occ rate
    uev_flat = occ_rate.values.flatten()
    uev_flat = uev_flat[~np.isnan(uev_flat)]
    if len(uev_flat) > 0:
        sns.kdeplot(uev_flat, ax=ax, color=COLORS['warning'], fill=True,
                    alpha=0.4, lw=2, label='UrbanEV Occ Rate', clip=(0, 1))
    ax.set_xlabel('Rate', fontsize=9)
    ax.set_ylabel('Density', fontsize=9)
    ax.set_title('(c) Utilization / Occupancy Distribution',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=8, framealpha=0.85)

    # ── (d) Revenue / Price distribution ────────────────────────
    ax = axes[1, 1]
    price_flat = urbanev_data['price'].values.flatten()
    price_flat = price_flat[~np.isnan(price_flat)]
    rev_data = acn_df['revenue_baseline'].dropna()

    ax.hist(rev_data, bins=50, color=COLORS['primary'], alpha=0.6,
            edgecolor='none', density=True, label='ACN Revenue (₹)', zorder=3)
    if len(price_flat) > 0:
        ax_twin = ax.twinx()
        ax_twin.hist(price_flat, bins=50, color=COLORS['warning'], alpha=0.5,
                     edgecolor='none', density=True, label='UrbanEV Price (¥)',
                     zorder=2)
        ax_twin.set_ylabel('UrbanEV Density', fontsize=9,
                           color=COLORS['warning'])
        h2, l2 = ax_twin.get_legend_handles_labels()
    else:
        h2, l2 = [], []
    ax.set_xlabel('Value', fontsize=9)
    ax.set_ylabel('ACN Density', fontsize=9, color=COLORS['primary'])
    ax.set_title('(d) Revenue / Price Distribution',
                 fontsize=12, fontweight='bold')
    h1, l1 = ax.get_legend_handles_labels()
    ax.legend(h1 + h2, l1 + l2, loc='upper right', fontsize=8,
              framealpha=0.85)

    fig.tight_layout(rect=[0, 0, 1, 0.95])
    save_figure(fig, '14_combined_demand_patterns.png')


# ═════════════════════════════════════════════════════════════════
#  SUMMARY STATISTICS
# ═════════════════════════════════════════════════════════════════

def _print_summary(acn_df, urbanev_data):
    print_header("EDA SUMMARY STATISTICS")

    # ── ACN ──
    print_subheader("ACN Dataset")
    n = len(acn_df)
    print(f"  Total sessions          : {n:,}")
    print(f"  Date range              : {acn_df['date'].min()} → {acn_df['date'].max()}")
    print(f"  Unique stations         : {acn_df['stationID'].nunique()}")
    print(f"  Sites                   : {', '.join(acn_df['site_label'].unique())}")
    print(f"  Avg kWh/session         : {acn_df['kWhDelivered'].mean():.2f}")
    print(f"  Median session (hrs)    : {acn_df['session_duration_hrs'].median():.2f}")
    print(f"  Mean utilization        : {acn_df['charger_utilization_rate'].mean():.3f}")
    print(f"  Mean idle time (hrs)    : {acn_df['idle_time_hrs'].mean():.2f}")
    print(f"  Total baseline revenue  : ₹{acn_df['revenue_baseline'].sum():,.0f}")

    # Period breakdown
    print_subheader("ACN — By TOU Period")
    period_stats = (acn_df.groupby('period')
                    .agg(sessions=('kWhDelivered', 'count'),
                         avg_kwh=('kWhDelivered', 'mean'),
                         avg_util=('charger_utilization_rate', 'mean'),
                         total_rev=('revenue_baseline', 'sum'))
                    .round(2))
    print(period_stats.to_string(index=True))

    # ── UrbanEV ──
    print_subheader("UrbanEV Dataset")
    occ = urbanev_data['occupancy']
    occ_rate = urbanev_data['occupancy_rate']
    ts = urbanev_data['timestamps']
    info = urbanev_data['information']
    stations = urbanev_data['stations']

    print(f"  Timestamp range         : {ts.min()} → {ts.max()}")
    print(f"  Number of timestamps    : {len(ts):,}")
    print(f"  Number of grids         : {occ.shape[1]}")
    print(f"  Number of stations      : {len(stations)}")
    if 'count' in stations.columns:
        print(f"  Total charger count     : {stations['count'].sum():,.0f}")
    print(f"  Mean occupancy rate     : {occ_rate.mean().mean():.4f}")
    print(f"  Max occupancy rate      : {occ_rate.max().max():.4f}")
    if 'CBD' in info.columns:
        cbd_count = int(info['CBD'].sum())
        print(f"  CBD grids               : {cbd_count} / {len(info)}")
    if 'dynamic_pricing' in info.columns:
        dp_count = int(info['dynamic_pricing'].sum())
        print(f"  Dynamic pricing grids   : {dp_count} / {len(info)}")

    print_subheader("EDA Complete — 14 Figures Saved")
    print(f"  Output directory: {FIGURES_DIR}")


# ═════════════════════════════════════════════════════════════════
#  MAIN ENTRY POINT
# ═════════════════════════════════════════════════════════════════

def run_eda(acn_df, urbanev_data):
    """Execute full Exploratory Data Analysis for OP'26 Analytics.

    Parameters
    ----------
    acn_df : pd.DataFrame
        Pre-processed ACN charging session data.
    urbanev_data : dict
        UrbanEV dataset dictionary with keys: occupancy, price, duration,
        volume, occupancy_rate, information, stations, timestamps.
    """
    apply_plot_style()
    print_header("EXPLORATORY DATA ANALYSIS")

    # ── ACN Plots (01-07) ──
    print_subheader("Generating ACN Plots")
    _plot_01_hourly_demand(acn_df)
    _plot_02_weekday_pattern(acn_df)
    _plot_03_kwh_distribution(acn_df)
    _plot_04_session_duration(acn_df)
    _plot_05_utilization_heatmap(acn_df)
    _plot_06_revenue_by_period(acn_df)
    _plot_07_idle_time(acn_df)

    # ── UrbanEV Plots (08-13) ──
    print_subheader("Generating UrbanEV Plots")
    _plot_08_urbanev_heatmap(urbanev_data)
    _plot_09_price_demand(urbanev_data)
    _plot_10_temporal_profile(urbanev_data)
    _plot_11_station_map(urbanev_data)
    _plot_12_peak_offpeak(urbanev_data)
    _plot_13_volatility(urbanev_data)

    # ── Combined Plot (14) ──
    print_subheader("Generating Combined Comparison")
    _plot_14_combined(acn_df, urbanev_data)

    # ── Summary ──
    _print_summary(acn_df, urbanev_data)


In [11]:
# ==================== 4. DEMAND PREDICTION AGENT ====================

"""
Demand Prediction Agent — EV Charging Tariff Optimization
==========================================================
Trains RandomForest and XGBoost regressors on ACN charger-utilization and
UrbanEV grid-occupancy datasets. Produces evaluation metrics, feature-
importance rankings, and five dark-mode plots.

Called as:
#     from src.demand_prediction_agent import DemandPredictionAgent
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor

# from src.utils import (
#     apply_plot_style, save_figure, save_metrics,
#     FIGURES_DIR, METRICS_DIR,
#     COLORS, PALETTE_CATEGORICAL,
#     print_header, print_subheader,
# )

# ────────────────────────────────────────────────────────────────
# CONSTANTS
# ────────────────────────────────────────────────────────────────
_ACN_FEATURES = [
    "hour", "day_of_week", "is_weekend", "month",
    "kWhDelivered_lag1", "queue_length_proxy",
]
_ACN_TARGET = "charger_utilization_rate"

_URBANEV_LAG_STEPS = [1, 2, 3, 6, 12]
_URBANEV_ROLLING_WIN = 12
_TOP_GRIDS = 20
_TRAIN_FRAC = 0.80
_RANDOM_STATE = 42


# ────────────────────────────────────────────────────────────────
# HELPER — build UrbanEV lag features for a single grid
# ────────────────────────────────────────────────────────────────
def _build_urbanev_features(series: pd.Series, timestamps: pd.DatetimeIndex) -> pd.DataFrame:
    """Create lag / rolling / calendar features from a grid occupancy-rate series."""
    df = pd.DataFrame({"occupancy_rate": series.values}, index=timestamps[: len(series)])
    for lag in _URBANEV_LAG_STEPS:
        df[f"lag_{lag}"] = df["occupancy_rate"].shift(lag)
    df["rolling_mean_12"] = df["occupancy_rate"].shift(1).rolling(_URBANEV_ROLLING_WIN).mean()
    df["hour"] = df.index.hour
    df["day_of_week"] = df.index.dayofweek
    df["is_weekend"] = (df["day_of_week"] >= 5).astype(int)
    df.dropna(inplace=True)
    return df


# ══════════════════════════════════════════════════════════════
# MAIN CLASS
# ══════════════════════════════════════════════════════════════
class DemandPredictionAgent:
    """Trains demand-prediction models on ACN & UrbanEV data,
    evaluates them, and persists plots + metrics."""

    def __init__(self):
        self.models: dict = {}          # model_name → fitted model
        self.results: dict = {}         # model_name → {y_test, y_pred, metrics, …}
        self._best_model_key: str = ""

    # ──────────────────────────────────────────────────────────
    # PUBLIC API
    # ──────────────────────────────────────────────────────────
    def train_and_evaluate(self, acn_df: pd.DataFrame, urbanev_data: dict) -> dict:
        """Train models on both datasets, evaluate, generate plots and metrics.

        Returns
        -------
        dict  with keys: predictions, metrics, feature_importances
        """
        print_header("DEMAND PREDICTION AGENT")

        # --- 1. ACN model ------------------------------------------------
        print_subheader("ACN — Charger Utilization Prediction")
        acn_metrics = self._train_acn(acn_df)

        # --- 2. UrbanEV model --------------------------------------------
        print_subheader("UrbanEV — Grid Occupancy Prediction")
        uev_metrics = self._train_urbanev(urbanev_data)

        # --- 3. Aggregate metrics & pick best model -----------------------
        all_metrics = {**acn_metrics, **uev_metrics}
        self._best_model_key = min(
            all_metrics, key=lambda k: all_metrics[k]["rmse"]
        )
        print(f"\n  ★ Best model (lowest RMSE): {self._best_model_key}")

        # --- 4. Generate all plots ----------------------------------------
        apply_plot_style()
        self._plot_actual_vs_predicted()     # 15
        self._plot_feature_importance()      # 16
        self._plot_residuals()               # 17
        self._plot_timeseries_forecast(urbanev_data)  # 18
        self._plot_model_comparison()        # 19

        # --- 5. Persist metrics CSV ---------------------------------------
        rows = []
        for name, m in all_metrics.items():
            rows.append({"model": name, "rmse": m["rmse"], "mae": m["mae"], "r2": m["r2"]})
        metrics_df = pd.DataFrame(rows).set_index("model")
        save_metrics(metrics_df, "demand_prediction_metrics.csv")

        return {
            "predictions": {k: v.get("y_pred") for k, v in self.results.items()},
            "metrics": all_metrics,
            "feature_importances": {
                k: v.get("feature_importances")
                for k, v in self.results.items()
                if v.get("feature_importances") is not None
            },
        }

    def predict(self, features_df: pd.DataFrame) -> np.ndarray:
        """Predict demand using the best available model."""
        if not self._best_model_key:
            raise RuntimeError("No trained model. Call train_and_evaluate() first.")
        model = self.models[self._best_model_key]
        return model.predict(features_df)

    # ──────────────────────────────────────────────────────────
    # INTERNAL — ACN training
    # ──────────────────────────────────────────────────────────
    def _train_acn(self, acn_df: pd.DataFrame) -> dict:
        df = acn_df.copy()

        # Create lag feature for kWhDelivered
        df = df.sort_values("connectionTime").reset_index(drop=True)
        df["kWhDelivered_lag1"] = df["kWhDelivered"].shift(1)
        df.dropna(subset=_ACN_FEATURES + [_ACN_TARGET], inplace=True)

        X = df[_ACN_FEATURES].values
        y = df[_ACN_TARGET].values
        feature_names = list(_ACN_FEATURES)

        split = int(len(X) * _TRAIN_FRAC)
        X_train, X_test = X[:split], X[split:]
        y_train, y_test = y[:split], y[split:]

        metrics_out = {}
        for tag, estimator in [
            ("ACN_RF", RandomForestRegressor(
                n_estimators=200, max_depth=12, min_samples_leaf=5,
                random_state=_RANDOM_STATE, n_jobs=-1)),
            ("ACN_XGB", XGBRegressor(
                n_estimators=200, max_depth=6, learning_rate=0.08,
                subsample=0.8, colsample_bytree=0.8,
                random_state=_RANDOM_STATE, verbosity=0)),
        ]:
            estimator.fit(X_train, y_train)
            y_pred = estimator.predict(X_test)
            rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
            mae = float(mean_absolute_error(y_test, y_pred))
            r2 = float(r2_score(y_test, y_pred))

            self.models[tag] = estimator
            fi = (estimator.feature_importances_
                  if hasattr(estimator, "feature_importances_") else None)
            self.results[tag] = {
                "y_test": y_test, "y_pred": y_pred,
                "metrics": {"rmse": rmse, "mae": mae, "r2": r2},
                "feature_names": feature_names,
                "feature_importances": (
                    pd.Series(fi, index=feature_names) if fi is not None else None
                ),
            }
            metrics_out[tag] = {"rmse": rmse, "mae": mae, "r2": r2}
            print(f"    {tag}  RMSE={rmse:.4f}  MAE={mae:.4f}  R²={r2:.4f}")

        return metrics_out

    # ──────────────────────────────────────────────────────────
    # INTERNAL — UrbanEV training
    # ──────────────────────────────────────────────────────────
    def _train_urbanev(self, urbanev_data: dict) -> dict:
        occ_rate = urbanev_data["occupancy_rate"]
        timestamps = urbanev_data["timestamps"]

        # Select top 20 grids by total occupancy
        total_occ = urbanev_data["occupancy"].sum(axis=0)
        top_grids = total_occ.nlargest(_TOP_GRIDS).index.tolist()

        # Collect rows from all top grids
        all_rows = []
        grid_indices: dict = {}  # grid → (start_idx, end_idx) into all_rows list
        for grid in top_grids:
            series = occ_rate[grid]
            gdf = _build_urbanev_features(series, timestamps)
            start = len(all_rows)
            all_rows.append(gdf)
            grid_indices[grid] = (start, start + len(gdf))

        combined = pd.concat(all_rows, ignore_index=False)
        feature_cols = [c for c in combined.columns if c != "occupancy_rate"]
        X = combined[feature_cols].values
        y = combined["occupancy_rate"].values
        feature_names = list(feature_cols)

        # Temporal split per concatenated block (first 80 % of each grid)
        train_mask = np.zeros(len(combined), dtype=bool)
        cumlen = 0
        for gdf in all_rows:
            n = len(gdf)
            cutoff = int(n * _TRAIN_FRAC)
            train_mask[cumlen: cumlen + cutoff] = True
            cumlen += n

        X_train, y_train = X[train_mask], y[train_mask]
        X_test, y_test = X[~train_mask], y[~train_mask]

        # Store combined DataFrame + test indices for time-series plot
        self._urbanev_combined = combined
        self._urbanev_test_mask = ~train_mask
        self._urbanev_top_grids = top_grids
        self._urbanev_all_rows = all_rows

        metrics_out = {}
        for tag, estimator in [
            ("UEV_RF", RandomForestRegressor(
                n_estimators=200, max_depth=14, min_samples_leaf=4,
                random_state=_RANDOM_STATE, n_jobs=-1)),
            ("UEV_XGB", XGBRegressor(
                n_estimators=250, max_depth=7, learning_rate=0.06,
                subsample=0.85, colsample_bytree=0.8,
                random_state=_RANDOM_STATE, verbosity=0)),
        ]:
            estimator.fit(X_train, y_train)
            y_pred = estimator.predict(X_test)
            rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
            mae = float(mean_absolute_error(y_test, y_pred))
            r2 = float(r2_score(y_test, y_pred))

            self.models[tag] = estimator
            fi = (estimator.feature_importances_
                  if hasattr(estimator, "feature_importances_") else None)
            self.results[tag] = {
                "y_test": y_test, "y_pred": y_pred,
                "metrics": {"rmse": rmse, "mae": mae, "r2": r2},
                "feature_names": feature_names,
                "feature_importances": (
                    pd.Series(fi, index=feature_names) if fi is not None else None
                ),
            }
            metrics_out[tag] = {"rmse": rmse, "mae": mae, "r2": r2}
            print(f"    {tag}  RMSE={rmse:.4f}  MAE={mae:.4f}  R²={r2:.4f}")

        return metrics_out

    # ──────────────────────────────────────────────────────────
    # PLOTS
    # ──────────────────────────────────────────────────────────
    def _plot_actual_vs_predicted(self):
        """15 — Scatter plot: actual vs predicted for the best model."""
        res = self.results[self._best_model_key]
        y_true, y_pred = res["y_test"], res["y_pred"]

        fig, ax = plt.subplots(figsize=(8, 7))
        ax.scatter(y_true, y_pred, alpha=0.35, s=18, color=COLORS["primary"],
                   edgecolors="none", label="Predictions")
        lims = [min(y_true.min(), y_pred.min()) - 0.02,
                max(y_true.max(), y_pred.max()) + 0.02]
        ax.plot(lims, lims, "--", color=COLORS["danger"], linewidth=1.5,
                label="Perfect Prediction")
        ax.set_xlim(lims)
        ax.set_ylim(lims)
        ax.set_xlabel("Actual")
        ax.set_ylabel("Predicted")
        ax.set_title(f"Actual vs Predicted — {self._best_model_key}")
        ax.legend()
        save_figure(fig, "15_demand_actual_vs_predicted.png")

    def _plot_feature_importance(self):
        """16 — Horizontal bar chart of top 15 features (best model)."""
        res = self.results[self._best_model_key]
        fi = res.get("feature_importances")
        if fi is None:
            return
        top15 = fi.sort_values(ascending=True).tail(15)

        fig, ax = plt.subplots(figsize=(8, 7))
        bars = ax.barh(top15.index, top15.values, color=COLORS["accent"], edgecolor="none")
        # Highlight the top feature
        bars[-1].set_color(COLORS["primary"])
        ax.set_xlabel("Importance")
        ax.set_title(f"Top 15 Feature Importances — {self._best_model_key}")
        save_figure(fig, "16_demand_feature_importance.png")

    def _plot_residuals(self):
        """17 — Residual distribution histogram for the best model."""
        res = self.results[self._best_model_key]
        residuals = res["y_test"] - res["y_pred"]

        fig, ax = plt.subplots(figsize=(8, 6))
        ax.hist(residuals, bins=50, color=COLORS["secondary"], edgecolor="#0f1117",
                alpha=0.85, density=True)
        ax.axvline(0, color=COLORS["danger"], linestyle="--", linewidth=1.4)
        ax.set_xlabel("Residual (Actual − Predicted)")
        ax.set_ylabel("Density")
        ax.set_title(f"Residual Distribution — {self._best_model_key}")

        # Annotate mean & std
        mu, sigma = residuals.mean(), residuals.std()
        ax.text(0.97, 0.95, f"μ = {mu:.4f}\nσ = {sigma:.4f}",
                transform=ax.transAxes, ha="right", va="top",
                fontsize=10, color=COLORS["accent"],
                bbox=dict(facecolor="#1a1d29", edgecolor=COLORS["accent"],
                          boxstyle="round,pad=0.4", alpha=0.9))
        save_figure(fig, "17_demand_residuals.png")

    def _plot_timeseries_forecast(self, urbanev_data: dict):
        """18 — Time series: actual vs predicted for 3 example grids."""
        # Pick the best UrbanEV model
        uev_key = "UEV_XGB" if "UEV_XGB" in self.models else "UEV_RF"
        model = self.models.get(uev_key)
        if model is None:
            return

        example_grids = self._urbanev_top_grids[:3]
        feature_cols = [c for c in self._urbanev_combined.columns if c != "occupancy_rate"]

        fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=False)
        palette = [COLORS["primary"], COLORS["accent"], COLORS["success"]]

        cumlen = 0
        for idx, (grid, gdf) in enumerate(zip(self._urbanev_top_grids, self._urbanev_all_rows)):
            n = len(gdf)
            if grid not in example_grids:
                cumlen += n
                continue
            ax_idx = example_grids.index(grid)
            ax = axes[ax_idx]

            cutoff = int(n * _TRAIN_FRAC)
            test_part = gdf.iloc[cutoff:]
            X_test_g = test_part[feature_cols].values
            y_pred_g = model.predict(X_test_g)

            ax.plot(test_part.index, test_part["occupancy_rate"].values,
                    color=COLORS["muted"], linewidth=1, label="Actual", alpha=0.8)
            ax.plot(test_part.index, y_pred_g,
                    color=palette[ax_idx], linewidth=1.2, label="Predicted", alpha=0.9)
            ax.set_title(f"Grid {grid}", fontsize=11)
            ax.legend(fontsize=9, loc="upper right")
            ax.set_ylabel("Occupancy Rate")
            cumlen += n

        axes[-1].set_xlabel("Timestamp")
        fig.suptitle("Time-Series Forecast — 3 Example Grids", fontsize=13, y=1.01)
        fig.tight_layout()
        save_figure(fig, "18_demand_timeseries_forecast.png")

    def _plot_model_comparison(self):
        """19 — Grouped bar chart comparing RF vs XGB across RMSE, MAE, R²."""
        metric_names = ["rmse", "mae", "r2"]
        model_keys = list(self.results.keys())

        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        x = np.arange(len(model_keys))
        width = 0.55

        for i, metric in enumerate(metric_names):
            vals = [self.results[k]["metrics"][metric] for k in model_keys]
            bars = axes[i].bar(x, vals, width, color=[PALETTE_CATEGORICAL[j % len(PALETTE_CATEGORICAL)]
                                                       for j in range(len(model_keys))],
                               edgecolor="none")
            axes[i].set_xticks(x)
            axes[i].set_xticklabels(model_keys, rotation=25, ha="right", fontsize=9)
            axes[i].set_title(metric.upper(), fontsize=12)

            # Annotate bars
            for bar, val in zip(bars, vals):
                axes[i].text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
                             f"{val:.3f}", ha="center", va="bottom", fontsize=9,
                             color=COLORS["accent"])

        fig.suptitle("Model Comparison — RF vs XGBoost", fontsize=13, y=1.02)
        fig.tight_layout()
        save_figure(fig, "19_demand_model_comparison.png")


In [12]:
# ==================== 5. TARIFF PRICING AGENT ====================

"""
Tariff Pricing Agent — EV Charging Tariff Optimization
=======================================================
Uses demand predictions from DemandPredictionAgent to generate dynamic
tariff multipliers, simulate revenue impact vs a flat ₹15/kWh baseline,
and quantify off-peak uplift via a simple demand-elasticity model.

Called as:
#     from src.tariff_pricing_agent import TariffPricingAgent
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# from src.utils import (
#     apply_plot_style, save_figure, save_metrics,
#     FIGURES_DIR, METRICS_DIR,
#     COLORS, PALETTE_CATEGORICAL,
#     BASELINE_TARIFF_PER_KWH,
#     SURGE_THRESHOLD, DISCOUNT_THRESHOLD,
#     SURGE_MULTIPLIER_RANGE, DISCOUNT_MULTIPLIER_RANGE,
#     print_header, print_subheader,
# )

# ────────────────────────────────────────────────────────────────
# CONSTANTS
# ────────────────────────────────────────────────────────────────
_PRICE_ELASTICITY = -0.30   # demand % change per price % change
_PERIOD_LABELS = {          # hour → pricing period
    "Off-Peak": list(range(0, 7)) + list(range(22, 24)),
    "Shoulder": list(range(7, 10)) + list(range(14, 17)),
    "Peak":     list(range(10, 14)) + list(range(17, 22)),
}


def _hour_to_period(h: int) -> str:
    for label, hours in _PERIOD_LABELS.items():
        if h in hours:
            return label
    return "Off-Peak"


def _compute_multiplier(util: float) -> float:
    """Map utilization to a tariff multiplier using linear interpolation."""
    if util >= SURGE_THRESHOLD:
        # Linearly scale 1.30→1.80 as util goes from 0.80→1.00
        frac = min((util - SURGE_THRESHOLD) / (1.0 - SURGE_THRESHOLD), 1.0)
        return SURGE_MULTIPLIER_RANGE[0] + frac * (SURGE_MULTIPLIER_RANGE[1] - SURGE_MULTIPLIER_RANGE[0])
    if util <= DISCOUNT_THRESHOLD:
        # Linearly scale 0.85→0.60 as util goes from 0.30→0.00
        frac = min((DISCOUNT_THRESHOLD - util) / DISCOUNT_THRESHOLD, 1.0)
        return DISCOUNT_MULTIPLIER_RANGE[1] - frac * (DISCOUNT_MULTIPLIER_RANGE[1] - DISCOUNT_MULTIPLIER_RANGE[0])
    return 1.0


# ══════════════════════════════════════════════════════════════
# MAIN CLASS
# ══════════════════════════════════════════════════════════════
class TariffPricingAgent:
    """Generates dynamic tariff multipliers and evaluates revenue impact."""

    def __init__(self, demand_agent):
        self.demand_agent = demand_agent
        self.pricing_history: list = []
        self.tariff_df: pd.DataFrame = pd.DataFrame()
        self.summary: dict = {}

    # ──────────────────────────────────────────────────────────
    # PUBLIC API
    # ──────────────────────────────────────────────────────────
    def optimize_tariffs(self, acn_df: pd.DataFrame, urbanev_data: dict) -> pd.DataFrame:
        """Generate dynamic tariff recommendations.

        Returns
        -------
        pd.DataFrame with columns:
            hour, day_of_week, period, predicted_util, tariff_multiplier,
            new_price, old_price, revenue_diff, kWhDelivered,
            sessions_old, sessions_new
        """
        print_header("TARIFF PRICING AGENT")

        # ----- 1. Build hourly utilization profile from ACN data ----------
        print_subheader("Building hourly utilization profile")
        tariff_df = self._build_hourly_profile(acn_df)

        # ----- 2. Apply pricing rules ------------------------------------
        print_subheader("Applying dynamic pricing rules")
        tariff_df["tariff_multiplier"] = tariff_df["predicted_util"].apply(_compute_multiplier)
        tariff_df["old_price"] = BASELINE_TARIFF_PER_KWH
        tariff_df["new_price"] = BASELINE_TARIFF_PER_KWH * tariff_df["tariff_multiplier"]
        tariff_df["period"] = tariff_df["hour"].apply(_hour_to_period)

        # ----- 3. Simulate demand elasticity response ---------------------
        print_subheader("Simulating demand elasticity")
        tariff_df = self._simulate_elasticity(tariff_df)

        # ----- 4. Compute revenue ----------------------------------------
        tariff_df["revenue_old"] = tariff_df["old_price"] * tariff_df["kWhDelivered"]
        tariff_df["revenue_new"] = tariff_df["new_price"] * tariff_df["kWhDelivered_adj"]
        tariff_df["revenue_diff"] = tariff_df["revenue_new"] - tariff_df["revenue_old"]

        self.tariff_df = tariff_df
        self.pricing_history.append(tariff_df.copy())

        # ----- 5. Compute summary metrics ---------------------------------
        self.summary = self._compute_summary(tariff_df)
        self._print_summary()

        # ----- 6. Plots & CSV output --------------------------------------
        apply_plot_style()
        self._plot_hourly_profile(tariff_df)           # 20
        self._plot_revenue_comparison(tariff_df)        # 21
        self._plot_utilization_shift(tariff_df)         # 22
        self._plot_heatmap(tariff_df)                   # 23
        self._plot_offpeak_uplift(tariff_df)            # 24

        # Metrics CSV
        metrics_rows = [
            {"metric": "Revenue Gain %", "value": self.summary["revenue_gain_pct"]},
            {"metric": "Avg Utilization Before", "value": self.summary["util_before"]},
            {"metric": "Avg Utilization After", "value": self.summary["util_after"]},
            {"metric": "Off-Peak Uplift %", "value": self.summary["offpeak_uplift_pct"]},
            {"metric": "Total Revenue Old (₹)", "value": self.summary["total_revenue_old"]},
            {"metric": "Total Revenue New (₹)", "value": self.summary["total_revenue_new"]},
        ]
        save_metrics(
            pd.DataFrame(metrics_rows).set_index("metric"),
            "tariff_optimization_metrics.csv",
        )

        # Recommendations CSV
        save_metrics(tariff_df, "tariff_recommendations.csv")

        return tariff_df

    # ──────────────────────────────────────────────────────────
    # INTERNAL — data preparation
    # ──────────────────────────────────────────────────────────
    def _build_hourly_profile(self, acn_df: pd.DataFrame) -> pd.DataFrame:
        """Aggregate ACN sessions into an hour × day_of_week profile."""
        profile = (
            acn_df.groupby(["hour", "day_of_week"])
            .agg(
                predicted_util=("charger_utilization_rate", "mean"),
                kWhDelivered=("kWhDelivered", "sum"),
                session_count=("kWhDelivered", "count"),
            )
            .reset_index()
        )
        return profile

    def _simulate_elasticity(self, df: pd.DataFrame) -> pd.DataFrame:
        """Apply price elasticity to estimate adjusted demand."""
        price_change_pct = (df["new_price"] - df["old_price"]) / df["old_price"]
        demand_change_pct = _PRICE_ELASTICITY * price_change_pct

        df["kWhDelivered_adj"] = df["kWhDelivered"] * (1 + demand_change_pct)
        df["sessions_old"] = df["session_count"]
        df["sessions_new"] = (df["session_count"] * (1 + demand_change_pct)).round().astype(int)
        df["util_after"] = df["predicted_util"] * (1 + demand_change_pct)
        df["util_after"] = df["util_after"].clip(0, 1)
        return df

    def _compute_summary(self, df: pd.DataFrame) -> dict:
        total_old = df["revenue_old"].sum()
        total_new = df["revenue_new"].sum()
        gain_pct = ((total_new - total_old) / total_old * 100) if total_old else 0

        offpeak = df[df["period"] == "Off-Peak"]
        offpeak_sessions_old = offpeak["sessions_old"].sum()
        offpeak_sessions_new = offpeak["sessions_new"].sum()
        offpeak_uplift = (
            ((offpeak_sessions_new - offpeak_sessions_old) / offpeak_sessions_old * 100)
            if offpeak_sessions_old else 0
        )

        return {
            "revenue_gain_pct": round(gain_pct, 2),
            "total_revenue_old": round(total_old, 2),
            "total_revenue_new": round(total_new, 2),
            "util_before": round(df["predicted_util"].mean(), 4),
            "util_after": round(df["util_after"].mean(), 4),
            "offpeak_uplift_pct": round(offpeak_uplift, 2),
        }

    def _print_summary(self):
        s = self.summary
        print(f"    Revenue Gain:          {s['revenue_gain_pct']:+.2f}%")
        print(f"    Avg Utilization Before: {s['util_before']:.4f}")
        print(f"    Avg Utilization After:  {s['util_after']:.4f}")
        print(f"    Off-Peak Uplift:        {s['offpeak_uplift_pct']:+.2f}%")

    # ──────────────────────────────────────────────────────────
    # PLOTS
    # ──────────────────────────────────────────────────────────
    def _plot_hourly_profile(self, df: pd.DataFrame):
        """20 — Dynamic tariff vs flat baseline across 24 hours."""
        hourly = df.groupby("hour").agg(
            avg_multiplier=("tariff_multiplier", "mean"),
        ).reset_index()
        hourly["dynamic_price"] = BASELINE_TARIFF_PER_KWH * hourly["avg_multiplier"]

        fig, ax = plt.subplots(figsize=(12, 5))
        ax.plot(hourly["hour"], hourly["dynamic_price"],
                color=COLORS["primary"], linewidth=2.2, marker="o", markersize=5,
                label="Dynamic Tariff (₹/kWh)")
        ax.axhline(BASELINE_TARIFF_PER_KWH, color=COLORS["danger"], linestyle="--",
                   linewidth=1.5, label=f"Flat Baseline ₹{BASELINE_TARIFF_PER_KWH}/kWh")

        # Shade surge / discount zones
        ax.fill_between(hourly["hour"], BASELINE_TARIFF_PER_KWH, hourly["dynamic_price"],
                        where=hourly["dynamic_price"] > BASELINE_TARIFF_PER_KWH,
                        interpolate=True, alpha=0.20, color=COLORS["danger"], label="Surge Zone")
        ax.fill_between(hourly["hour"], hourly["dynamic_price"], BASELINE_TARIFF_PER_KWH,
                        where=hourly["dynamic_price"] < BASELINE_TARIFF_PER_KWH,
                        interpolate=True, alpha=0.20, color=COLORS["success"], label="Discount Zone")

        ax.set_xlabel("Hour of Day")
        ax.set_ylabel("Tariff (₹/kWh)")
        ax.set_title("Hourly Dynamic Tariff Profile vs Flat Baseline")
        ax.set_xticks(range(24))
        ax.legend(fontsize=9)
        save_figure(fig, "20_tariff_hourly_profile.png")

    def _plot_revenue_comparison(self, df: pd.DataFrame):
        """21 — Grouped bar chart: revenue by period, old vs new."""
        period_rev = df.groupby("period").agg(
            old=("revenue_old", "sum"),
            new=("revenue_new", "sum"),
        )
        # Ensure consistent period order
        order = ["Off-Peak", "Shoulder", "Peak"]
        period_rev = period_rev.reindex([p for p in order if p in period_rev.index])

        fig, ax = plt.subplots(figsize=(9, 6))
        x = np.arange(len(period_rev))
        w = 0.35
        ax.bar(x - w / 2, period_rev["old"], w, label="Flat Baseline",
               color=COLORS["muted"], edgecolor="none")
        ax.bar(x + w / 2, period_rev["new"], w, label="Dynamic Tariff",
               color=COLORS["primary"], edgecolor="none")

        ax.set_xticks(x)
        ax.set_xticklabels(period_rev.index)
        ax.set_ylabel("Revenue (₹)")
        ax.set_title("Revenue by Period — Flat vs Dynamic Tariff")
        ax.legend()

        # Annotate % change
        for i, period in enumerate(period_rev.index):
            old, new = period_rev.loc[period, "old"], period_rev.loc[period, "new"]
            pct = ((new - old) / old * 100) if old else 0
            color = COLORS["success"] if pct >= 0 else COLORS["danger"]
            ax.text(i + w / 2, new, f"{pct:+.1f}%", ha="center", va="bottom",
                    fontsize=9, color=color, fontweight="bold")
        save_figure(fig, "21_tariff_revenue_comparison.png")

    def _plot_utilization_shift(self, df: pd.DataFrame):
        """22 — Before / after utilization distribution (histogram overlay)."""
        fig, ax = plt.subplots(figsize=(9, 6))
        bins = np.linspace(0, 1, 40)
        ax.hist(df["predicted_util"], bins=bins, alpha=0.55,
                color=COLORS["muted"], edgecolor="#0f1117", label="Before (Flat)")
        ax.hist(df["util_after"], bins=bins, alpha=0.55,
                color=COLORS["primary"], edgecolor="#0f1117", label="After (Dynamic)")
        ax.set_xlabel("Charger Utilization Rate")
        ax.set_ylabel("Count")
        ax.set_title("Utilization Distribution — Before vs After Dynamic Pricing")
        ax.legend()
        save_figure(fig, "22_tariff_utilization_shift.png")

    def _plot_heatmap(self, df: pd.DataFrame):
        """23 — Heatmap: recommended tariff multiplier (hour × day_of_week)."""
        pivot = df.pivot_table(
            values="tariff_multiplier", index="day_of_week", columns="hour",
            aggfunc="mean",
        )
        day_labels = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
        pivot.index = [day_labels[i] if i < len(day_labels) else str(i)
                       for i in pivot.index]

        fig, ax = plt.subplots(figsize=(14, 5))
        sns.heatmap(
            pivot, ax=ax, cmap="RdYlGn_r", center=1.0,
            linewidths=0.4, linecolor="#2d3148",
            cbar_kws={"label": "Tariff Multiplier"},
            annot=True, fmt=".2f", annot_kws={"fontsize": 8},
        )
        ax.set_title("Recommended Tariff Multipliers (Hour × Day)")
        ax.set_xlabel("Hour of Day")
        ax.set_ylabel("")
        save_figure(fig, "23_tariff_heatmap.png")

    def _plot_offpeak_uplift(self, df: pd.DataFrame):
        """24 — Bar chart: session count increase in off-peak periods."""
        offpeak = df[df["period"] == "Off-Peak"].copy()
        grouped = offpeak.groupby("hour").agg(
            before=("sessions_old", "sum"),
            after=("sessions_new", "sum"),
        ).reset_index()

        fig, ax = plt.subplots(figsize=(10, 5))
        x = np.arange(len(grouped))
        w = 0.35
        ax.bar(x - w / 2, grouped["before"], w, label="Before (Flat)",
               color=COLORS["muted"], edgecolor="none")
        ax.bar(x + w / 2, grouped["after"], w, label="After (Dynamic)",
               color=COLORS["success"], edgecolor="none")

        ax.set_xticks(x)
        ax.set_xticklabels(grouped["hour"].astype(str))
        ax.set_xlabel("Hour of Day (Off-Peak)")
        ax.set_ylabel("Session Count")
        ax.set_title("Off-Peak Session Uplift with Discount Pricing")
        ax.legend()
        save_figure(fig, "24_tariff_offpeak_uplift.png")


In [13]:
# ==================== 6. MONITORING & LEARNING AGENT ====================

"""
Monitoring & Learning Agent — EV Charging Tariff Optimization
==============================================================
Runs an iterative feedback loop that alternates between demand
prediction, tariff setting, and simulated customer response.
Tracks revenue, utilization, wait-time proxy, and pricing
efficiency across episodes, then persists plots and metrics.

Called as:
#     from src.monitoring_agent import MonitoringAgent
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# from src.utils import (
#     apply_plot_style, save_figure, save_metrics,
#     FIGURES_DIR, METRICS_DIR,
#     COLORS, PALETTE_CATEGORICAL,
#     BASELINE_TARIFF_PER_KWH,
#     SURGE_THRESHOLD, DISCOUNT_THRESHOLD,
#     SURGE_MULTIPLIER_RANGE, DISCOUNT_MULTIPLIER_RANGE,
#     print_header, print_subheader,
# )

# ────────────────────────────────────────────────────────────────
# CONSTANTS
# ────────────────────────────────────────────────────────────────
_BASE_ELASTICITY = 0.30
_LEARNING_RATE = 0.04        # agent parameter adjustment step
_WAIT_TIME_SCALE = 2.5       # minutes of wait per unit excess utilization
_MIN_MULTIPLIER = 0.50
_MAX_MULTIPLIER = 2.00


def _compute_multiplier(util: float) -> float:
    """Identical pricing logic to TariffPricingAgent for consistency."""
    if util >= SURGE_THRESHOLD:
        frac = min((util - SURGE_THRESHOLD) / (1.0 - SURGE_THRESHOLD), 1.0)
        return SURGE_MULTIPLIER_RANGE[0] + frac * (SURGE_MULTIPLIER_RANGE[1] - SURGE_MULTIPLIER_RANGE[0])
    if util <= DISCOUNT_THRESHOLD:
        frac = min((DISCOUNT_THRESHOLD - util) / DISCOUNT_THRESHOLD, 1.0)
        return DISCOUNT_MULTIPLIER_RANGE[1] - frac * (DISCOUNT_MULTIPLIER_RANGE[1] - DISCOUNT_MULTIPLIER_RANGE[0])
    return 1.0


# ══════════════════════════════════════════════════════════════
# MAIN CLASS
# ══════════════════════════════════════════════════════════════
class MonitoringAgent:
    """Iterative feedback loop that refines pricing across episodes."""

    def __init__(self, demand_agent, tariff_agent):
        self.demand_agent = demand_agent
        self.tariff_agent = tariff_agent
        self.episode_history: list = []

    # ──────────────────────────────────────────────────────────
    # PUBLIC API
    # ──────────────────────────────────────────────────────────
    def run_feedback_loop(
        self, acn_df: pd.DataFrame, urbanev_data: dict, n_episodes: int = 10
    ) -> pd.DataFrame:
        """Run *n_episodes* of pricing → outcome → adjustment cycles.

        Returns
        -------
        pd.DataFrame  one row per episode with tracked KPIs.
        """
        print_header("MONITORING & LEARNING AGENT")

        # Build a base hourly profile from ACN data
        base_profile = self._build_base_profile(acn_df)

        # Agent-adjustable parameters that evolve across episodes
        elasticity = _BASE_ELASTICITY
        surge_bonus = 0.0    # additive tweak to surge multiplier
        discount_bonus = 0.0 # additive tweak to discount depth

        for ep in range(1, n_episodes + 1):
            print_subheader(f"Episode {ep}/{n_episodes}")
            record = self._run_single_episode(
                base_profile, ep, elasticity, surge_bonus, discount_bonus
            )

            # ---- Adjustment heuristics ----------------------------------
            # If utilization is still too high on average, increase surge
            if record["avg_utilization"] > 0.75:
                surge_bonus = min(surge_bonus + _LEARNING_RATE, 0.30)
            elif record["avg_utilization"] < 0.45:
                surge_bonus = max(surge_bonus - _LEARNING_RATE * 0.5, -0.10)

            # If off-peak is underserved, deepen discount
            if record["offpeak_util"] < 0.35:
                discount_bonus = min(discount_bonus + _LEARNING_RATE, 0.20)
            else:
                discount_bonus = max(discount_bonus - _LEARNING_RATE * 0.5, -0.05)

            # Elasticity slowly converges toward observed responsiveness
            observed_response = abs(record["volume_change_pct"]) / max(abs(record["price_change_pct"]), 1e-6)
            elasticity = 0.9 * elasticity + 0.1 * observed_response

            self.episode_history.append(record)
            print(f"    Rev=₹{record['total_revenue']:,.0f}  "
                  f"Util={record['avg_utilization']:.3f}  "
                  f"Wait={record['avg_wait_time_proxy']:.2f}min  "
                  f"Eff={record['pricing_efficiency']:.3f}")

        # ---- Aggregate into DataFrame -----------------------------------
        history_df = pd.DataFrame(self.episode_history)

        # ---- Evaluation metrics -----------------------------------------
        print_subheader("Episode Summary Metrics")
        wait_reduction = (
            ((history_df["avg_wait_time_proxy"].iloc[0] -
              history_df["avg_wait_time_proxy"].iloc[-1]) /
             history_df["avg_wait_time_proxy"].iloc[0] * 100)
            if history_df["avg_wait_time_proxy"].iloc[0] > 0 else 0
        )
        avg_response = history_df["volume_change_pct"].abs().mean()
        final_efficiency = history_df["pricing_efficiency"].iloc[-1]

        print(f"    Avg Waiting-Time Reduction: {wait_reduction:+.2f}%")
        print(f"    Customer Response Rate:     {avg_response:.4f}")
        print(f"    Final Pricing Efficiency:   {final_efficiency:.4f}")

        # ---- Plots -------------------------------------------------------
        apply_plot_style()
        self._plot_learning_curve_revenue(history_df)        # 25
        self._plot_learning_curve_utilization(history_df)     # 26
        self._plot_wait_time_reduction(history_df)            # 27
        self._plot_pricing_efficiency(history_df)             # 28
        self._plot_feedback_summary(history_df)               # 29

        # ---- Metrics CSV --------------------------------------------------
        metrics_rows = [
            {"metric": "Avg Waiting Time Reduction %", "value": round(wait_reduction, 2)},
            {"metric": "Customer Response Rate", "value": round(avg_response, 4)},
            {"metric": "Final Pricing Efficiency", "value": round(final_efficiency, 4)},
            {"metric": "Final Avg Utilization", "value": round(history_df["avg_utilization"].iloc[-1], 4)},
            {"metric": "Final Revenue (₹)", "value": round(history_df["total_revenue"].iloc[-1], 2)},
        ]
        save_metrics(
            pd.DataFrame(metrics_rows).set_index("metric"),
            "monitoring_metrics.csv",
        )

        return history_df

    # ──────────────────────────────────────────────────────────
    # INTERNAL — single episode
    # ──────────────────────────────────────────────────────────
    def _build_base_profile(self, acn_df: pd.DataFrame) -> pd.DataFrame:
        """Aggregate ACN data into an hour × day_of_week base profile."""
        profile = (
            acn_df.groupby(["hour", "day_of_week"])
            .agg(
                base_util=("charger_utilization_rate", "mean"),
                kWhDelivered=("kWhDelivered", "sum"),
                session_count=("kWhDelivered", "count"),
            )
            .reset_index()
        )
        return profile

    def _run_single_episode(
        self, base_profile: pd.DataFrame, episode: int,
        elasticity: float, surge_bonus: float, discount_bonus: float,
    ) -> dict:
        """Execute one pricing→outcome cycle and return a record dict."""
        df = base_profile.copy()

        # Add small noise to simulate episode variability
        rng = np.random.default_rng(seed=42 + episode)
        noise = rng.normal(0, 0.02, size=len(df))
        df["util"] = (df["base_util"] + noise).clip(0, 1)

        # Compute tariff multipliers with agent adjustments
        multipliers = []
        for u in df["util"]:
            m = _compute_multiplier(u)
            if m > 1.0:
                m = min(m + surge_bonus, _MAX_MULTIPLIER)
            elif m < 1.0:
                m = max(m - discount_bonus, _MIN_MULTIPLIER)
            multipliers.append(m)
        df["multiplier"] = multipliers
        df["new_price"] = BASELINE_TARIFF_PER_KWH * df["multiplier"]
        df["old_price"] = BASELINE_TARIFF_PER_KWH

        # Elasticity response
        price_change_pct = (df["new_price"] - df["old_price"]) / df["old_price"]
        demand_change_pct = -elasticity * price_change_pct
        df["kWh_adj"] = df["kWhDelivered"] * (1 + demand_change_pct)
        df["util_adj"] = (df["util"] * (1 + demand_change_pct)).clip(0, 1)

        # Revenue
        df["revenue"] = df["new_price"] * df["kWh_adj"]

        # Wait-time proxy: if util > 0.90, excess → wait
        df["wait_proxy"] = np.where(
            df["util_adj"] > 0.90,
            (df["util_adj"] - 0.90) * _WAIT_TIME_SCALE * 60,  # in minutes
            0,
        )

        # Period tagging
        def _period(h):
            if h in list(range(0, 7)) + list(range(22, 24)):
                return "Off-Peak"
            if h in list(range(10, 14)) + list(range(17, 22)):
                return "Peak"
            return "Shoulder"

        df["period"] = df["hour"].apply(_period)
        offpeak = df[df["period"] == "Off-Peak"]

        total_revenue = df["revenue"].sum()
        total_kwh = df["kWh_adj"].sum()

        return {
            "episode": episode,
            "total_revenue": total_revenue,
            "avg_utilization": df["util_adj"].mean(),
            "avg_wait_time_proxy": df["wait_proxy"].mean(),
            "pricing_efficiency": total_revenue / max(total_kwh, 1.0),
            "offpeak_util": offpeak["util_adj"].mean() if len(offpeak) else 0,
            "price_change_pct": price_change_pct.mean(),
            "volume_change_pct": demand_change_pct.mean(),
        }

    # ──────────────────────────────────────────────────────────
    # PLOTS
    # ──────────────────────────────────────────────────────────
    def _plot_learning_curve_revenue(self, df: pd.DataFrame):
        """25 — Revenue across episodes."""
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.plot(df["episode"], df["total_revenue"],
                color=COLORS["primary"], marker="o", markersize=6, linewidth=2)
        ax.fill_between(df["episode"], df["total_revenue"],
                        alpha=0.15, color=COLORS["primary"])
        ax.set_xlabel("Episode")
        ax.set_ylabel("Total Revenue (₹)")
        ax.set_title("Learning Curve — Revenue Across Episodes")
        ax.set_xticks(df["episode"])
        save_figure(fig, "25_learning_curve_revenue.png")

    def _plot_learning_curve_utilization(self, df: pd.DataFrame):
        """26 — Utilization across episodes."""
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.plot(df["episode"], df["avg_utilization"],
                color=COLORS["accent"], marker="s", markersize=6, linewidth=2)
        ax.axhline(0.70, color=COLORS["warning"], linestyle="--", linewidth=1,
                   label="Target (0.70)")
        ax.fill_between(df["episode"], df["avg_utilization"],
                        alpha=0.15, color=COLORS["accent"])
        ax.set_xlabel("Episode")
        ax.set_ylabel("Avg Utilization")
        ax.set_title("Learning Curve — Utilization Across Episodes")
        ax.set_xticks(df["episode"])
        ax.legend()
        save_figure(fig, "26_learning_curve_utilization.png")

    def _plot_wait_time_reduction(self, df: pd.DataFrame):
        """27 — Wait-time proxy across episodes (bar chart)."""
        fig, ax = plt.subplots(figsize=(10, 5))
        colors = [COLORS["danger"] if w > df["avg_wait_time_proxy"].median()
                  else COLORS["success"] for w in df["avg_wait_time_proxy"]]
        ax.bar(df["episode"], df["avg_wait_time_proxy"],
               color=colors, edgecolor="none", width=0.7)
        ax.set_xlabel("Episode")
        ax.set_ylabel("Avg Wait-Time Proxy (min)")
        ax.set_title("Wait-Time Proxy Across Episodes")
        ax.set_xticks(df["episode"])
        save_figure(fig, "27_wait_time_reduction.png")

    def _plot_pricing_efficiency(self, df: pd.DataFrame):
        """28 — Pricing efficiency (₹/kWh) across episodes."""
        fig, ax = plt.subplots(figsize=(10, 5))
        ax.plot(df["episode"], df["pricing_efficiency"],
                color=COLORS["highlight"], marker="D", markersize=6, linewidth=2)
        ax.fill_between(df["episode"], df["pricing_efficiency"],
                        alpha=0.12, color=COLORS["highlight"])
        ax.axhline(BASELINE_TARIFF_PER_KWH, color=COLORS["muted"], linestyle="--",
                   linewidth=1, label=f"Baseline ₹{BASELINE_TARIFF_PER_KWH}/kWh")
        ax.set_xlabel("Episode")
        ax.set_ylabel("Pricing Efficiency (₹/kWh)")
        ax.set_title("Pricing Efficiency Trajectory")
        ax.set_xticks(df["episode"])
        ax.legend()
        save_figure(fig, "28_pricing_efficiency_trajectory.png")

    def _plot_feedback_summary(self, df: pd.DataFrame):
        """29 — 2×2 subplot combining all four KPIs across episodes."""
        fig = plt.figure(figsize=(14, 10))
        gs = gridspec.GridSpec(2, 2, hspace=0.35, wspace=0.30)

        configs = [
            ("total_revenue", "Revenue (₹)", COLORS["primary"],
             "Revenue", "o"),
            ("avg_utilization", "Avg Utilization", COLORS["accent"],
             "Utilization", "s"),
            ("avg_wait_time_proxy", "Wait-Time Proxy (min)", COLORS["danger"],
             "Wait Time", "^"),
            ("pricing_efficiency", "Pricing Efficiency (₹/kWh)", COLORS["highlight"],
             "Efficiency", "D"),
        ]

        for idx, (col, ylabel, color, title, marker) in enumerate(configs):
            ax = fig.add_subplot(gs[idx])
            ax.plot(df["episode"], df[col], color=color,
                    marker=marker, markersize=5, linewidth=1.8)
            ax.fill_between(df["episode"], df[col], alpha=0.12, color=color)
            ax.set_xlabel("Episode")
            ax.set_ylabel(ylabel)
            ax.set_title(title, fontsize=12)
            ax.set_xticks(df["episode"])

        fig.suptitle("Feedback Loop Summary — All KPIs", fontsize=14, y=1.01)
        save_figure(fig, "29_feedback_loop_summary.png")


In [14]:
# ==================== 7. PIPELINE EXECUTION ====================

# 1. Run Preprocessing
acn_df, urbanev_data = run_preprocessing()

# 2. Run Exploratory Data Analysis
run_eda(acn_df, urbanev_data)

# 3. Train Demand Forecasting Models
demand_agent = DemandPredictionAgent()
demand_results = demand_agent.train_and_evaluate(acn_df, urbanev_data)

# 4. Run Dynamic Pricing Optimization
tariff_agent = TariffPricingAgent(demand_agent)
tariff_results = tariff_agent.optimize_tariffs(acn_df, urbanev_data)

# 5. Run Self-Improving Feedback Loop
monitor = MonitoringAgent(demand_agent, tariff_agent)
monitor_results = monitor.run_feedback_loop(acn_df, urbanev_data, n_episodes=10)



  DATA PREPROCESSING

  -- Loading ACN-Data (Caltech/JPL sessions) --
  Raw records: 16,304
  Columns: ['_meta', 'end', 'min_kWh', 'site', 'start', '_items', '_id', 'clusterID', 'connectionTime', 'disconnectTime', 'doneChargingTime', 'kWhDelivered', 'sessionID', 'siteID', 'spaceID', 'stationID', 'timezone', 'userID', 'userInputs', 'WhPerMile', 'kWhRequested', 'milesRequested', 'minutesAvailable', 'modifiedAt', 'paymentRequired', 'requestedDeparture', 'userID.1']

  -- Preprocessing ACN-Data --
  Dropped 1305 rows with missing connection/disconnect times
  Filtered 52 impossible-duration sessions
  Final ACN records: 14,947
  Date range: 2018-04-25 to 2018-12-16
  Sites: ['caltech' 'unknown']
  Unique stations: 54
  Avg kWh/session: 9.00
  Avg session duration: 5.68 hrs
  Avg utilization rate: 69.34%

  -- Loading UrbanEV Dataset (Shenzhen) --
  occupancy.csv: 8,640 timestamps × 247 grids
  price.csv: 8,640 timestamps × 247 grids
  duration.csv: 8,640 timestamps × 247 grids
  volume.cs

## Visualizations and Submission Results

All model results, dynamic schedules, and feedback outcomes are successfully computed and printed above.

* **Plots Generated:** Saved directly under `outputs/figures/` (viewable interactively via `dashboard/index.html`).
* **Metrics CSVs:** Saved under `outputs/metrics/` for final submission.